# 01 · Data quality – Nordvia Retail (SAP FI)

**Goal.** Assess the quality of the 11 CSV tables before designing the ontology, and leave a reproducible **data quality log** (deliverable 5 of the challenge).

**Scope of this notebook**
1. Load the tables and the data dictionary.
2. Profiling: completeness, primary-key uniqueness and consistency with the dictionary.
3. Referential integrity: join every child table to its parent table by key and count how many rows are left without a match.
4. Skeleton of the data quality log (findings with the decision **pending**; design decisions are taken by the author).

**Dataset conventions** (README): identifiers have leading zeros, so they are read as text; dates use `YYYYMMDD`; amounts are always positive and the sign comes from `SHKZG`.

## 0 · Environment

In [1]:
import os
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

In [2]:
%pip install pandas

import os, glob, sys
import pandas as pd


pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)

print("Python", sys.version.split()[0], "| pandas", pd.__version__)

DATA_PATH = "data"
DICT_PATH = "DATA_DICTIONARY.csv"
print("Data:", os.path.abspath(DATA_PATH))

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.2 -> 26.3
[notice] To update, run: c:\Users\Javier Granda\.pyenv\pyenv-win\versions\3.12.7\python.exe -m pip install --upgrade pip


Python 3.12.7 | pandas 3.1.1
Data: c:\Users\Javier Granda\Desktop\tech-nordvia-challenge-candidate-pack\submission\data


## 1 · Loading
Everything is read as **text** (`dtype=str`) and empty strings are not converted to `NaN`. This keeps leading zeros and lets us tell "empty" apart from "missing". Typing (dates, decimals) is done later, in the cleaning step.

In [3]:
tables = {os.path.basename(f)[:-4]: pd.read_csv(f, dtype=str, keep_default_na=False, encoding="utf-8")
          for f in sorted(glob.glob(os.path.join(DATA_PATH, "*.csv")))}
dictionary = pd.read_csv(DICT_PATH, dtype=str, encoding="utf-8")

summary = {**tables, "DATA_DICTIONARY": dictionary}
pd.DataFrame({"rows": {t: len(d) for t, d in summary.items()},
              "columns": {t: d.shape[1] for t, d in summary.items()}})

,rows,columns
BKPF,2353,15
BSEG,6062,19
CSKS,25,9
KNA1,156,10
LFA1,72,10
SKA1,27,7
SKAT,55,5
SKB1,80,7
T001,3,8
T003,7,3


In [4]:
mismatches = []
for t, d in tables.items():
    expected = list(dictionary.loc[dictionary["table"] == t, "field"])
    extra, missing = set(d.columns) - set(expected), set(expected) - set(d.columns)
    if extra or missing:
        mismatches.append((t, extra, missing))
print("Fields in the dictionary:", len(dictionary))
print("Tables that differ from the dictionary:", mismatches or "none")

Fields in the dictionary: 97
Tables that differ from the dictionary: none


## 2 · Profiling
### 2.1 Completeness and cardinality per column
Blank strings count as empty. An empty value is **not always an error** (e.g. `KUNNR` is only filled on customer lines), so this step only measures; interpretation comes later.

In [5]:
def profile(d):
    rows = []
    for pos, c in enumerate(d.columns, start=1):
        s = d[c]
        rows.append({"column_position": pos, "column": c,
                     "empty_count": int((s.str.strip() == "").sum()),
                     "with_spaces": int((s != s.str.strip()).sum()),
                     "unique": s.nunique()})
    p = pd.DataFrame(rows)
    p["pct_empty"] = (100 * p["empty_count"] / len(d)).round(2)
    return p

profiles = (pd.concat({t: profile(d) for t, d in tables.items()}, names=["table"])
              .reset_index(level=0).reset_index(drop=True))
profiles[(profiles["empty_count"] > 0) | (profiles["with_spaces"] > 0)].reset_index(drop=True)

,table,column_position,column,empty_count,with_spaces,unique,pct_empty
0,BKPF,12,XBLNR,155,0,1657,6.59
1,BKPF,13,BKTXT,2198,0,52,93.41
2,BKPF,14,STBLG,2339,0,12,99.41
3,BKPF,15,STJAH,2339,0,2,99.41
4,BSEG,9,KUNNR,4763,0,143,78.57
5,BSEG,10,LIFNR,5156,0,73,85.05
6,BSEG,13,MWSKZ,4198,0,7,69.25
7,BSEG,14,KOSTL,5025,0,26,82.89
8,BSEG,15,ZTERM,4869,0,6,80.32
9,BSEG,16,ZFBDT,4869,0,358,80.32


### 2.2 Primary-key uniqueness (according to the dictionary)

In [6]:
out = []
for t, d in tables.items():
    pk = list(dictionary.loc[(dictionary["table"] == t) & (dictionary.is_primary_key == "Y"), "field"])
    out.append({"table": t, "key": " + ".join(pk),
                "rows": len(d), "duplicated_keys": int(d.duplicated(pk).sum()),
                "fully_duplicated_rows": int(d.duplicated().sum())})
pd.DataFrame(out)

,table,key,rows,duplicated_keys,fully_duplicated_rows
0,BKPF,BUKRS + BELNR + GJAHR,2353,0,0
1,BSEG,BUKRS + BELNR + GJAHR + BUZEI,6062,0,0
2,CSKS,KOKRS + KOSTL + DATBI,25,0,0
3,KNA1,KUNNR,156,0,0
4,LFA1,LIFNR,72,0,0
5,SKA1,KTOPL + SAKNR,27,0,0
6,SKAT,SPRAS + KTOPL + SAKNR,55,0,0
7,SKB1,BUKRS + SAKNR,80,0,0
8,T001,BUKRS,3,0,0
9,T003,BLART,7,0,0


## 3 · Referential integrity: joining by keys
**Idea.** A primary key shows up in another table (as a *foreign key*) only when the rows of that table refer to a row of the first one. Every value of the foreign key must exist in the parent table. Here we measure, for each relationship, how many rows of the child table **find no parent**.

- Only rows where the key is filled are considered (e.g. in `BSEG`, an empty `KUNNR` means the line has no customer).
- Composite keys are joined on **all** their columns at once.
- `parents_without_children` is informational and is rarely an error (e.g. customers with no postings).

In [7]:
def check_join(child, parent, ck, pk=None):
    pk = pk or ck
    C = tables[child]
    P = tables[parent][pk].drop_duplicates().set_axis(ck, axis=1)
    with_key = C[(C[ck] != "").all(axis=1)]
    m = with_key.merge(P, on=ck, how="left", indicator=True)
    orphans = m[m["_merge"] == "left_only"]
    used = with_key[ck].drop_duplicates()
    unused = P.merge(used, on=ck, how="left", indicator=True)["_merge"].eq("left_only").sum()
    return dict(child=child, parent=parent, key="+".join(ck),
                rows_with_key=len(with_key), rows_without_parent=len(orphans),
                pct_without_parent=round(100 * len(orphans) / len(with_key), 2),
                example="|".join(orphans[ck].iloc[0]) if len(orphans) else None,
                parents_without_children=int(unused))


RELATIONSHIPS = [
    ("BKPF", "T001", ["BUKRS"]),
    ("BKPF", "T003", ["BLART"]),
    ("BSEG", "BKPF", ["BUKRS", "BELNR", "GJAHR"]),
    ("BSEG", "T001", ["BUKRS"]),
    ("BSEG", "TBSL", ["BSCHL"]),
    ("BSEG", "KNA1", ["KUNNR"]),
    ("BSEG", "LFA1", ["LIFNR"]),
    ("BSEG", "SKA1", ["HKONT"], ["SAKNR"]),
    ("BSEG", "SKB1", ["BUKRS", "HKONT"], ["BUKRS", "SAKNR"]),
    ("BSEG", "CSKS", ["KOSTL"]),
    ("BSEG", "BKPF", ["BUKRS", "AUGBL"], ["BUKRS", "BELNR"]),                       
    ("BKPF", "BKPF", ["BUKRS", "STBLG", "STJAH"], ["BUKRS", "BELNR", "GJAHR"]),     
    ("SKB1", "T001", ["BUKRS"]),
    ("SKB1", "SKA1", ["SAKNR"]),
    ("SKAT", "SKA1", ["KTOPL", "SAKNR"]),
    ("SKA1", "SKAT", ["KTOPL", "SAKNR"]),                                           
    ("CSKS", "T001", ["BUKRS"]),
    ("T001", "SKA1", ["KTOPL"]),
]
joins = pd.DataFrame([check_join(*r) for r in RELATIONSHIPS])
joins

,child,parent,key,rows_with_key,rows_without_parent,pct_without_parent,example,parents_without_children
0,BKPF,T001,BUKRS,2353,0,0.00,NaN,0
1,BKPF,T003,BLART,2353,0,0.00,NaN,0
2,BSEG,BKPF,BUKRS+BELNR+GJAHR,6062,0,0.00,NaN,1
3,BSEG,T001,BUKRS,6062,0,0.00,NaN,0
4,BSEG,TBSL,BSCHL,6062,0,0.00,NaN,0
5,BSEG,KNA1,KUNNR,1299,3,0.23,0000100999,15
6,BSEG,LFA1,LIFNR,906,0,0.00,NaN,0
7,BSEG,SKA1,HKONT,6062,0,0.00,NaN,2
8,BSEG,SKB1,BUKRS+HKONT,6062,2,0.03,3000|0000800100,10
9,BSEG,CSKS,KOSTL,1037,0,0.00,NaN,0


### Relationships with unmatched rows

In [8]:
joins[joins.rows_without_parent > 0]

,child,parent,key,rows_with_key,rows_without_parent,pct_without_parent,example,parents_without_children
5,BSEG,KNA1,KUNNR,1299,3,0.23,0000100999,15
8,BSEG,SKB1,BUKRS+HKONT,6062,2,0.03,3000|0000800100,10
14,SKAT,SKA1,KTOPL+SAKNR,55,2,3.64,NVCA|0000455000,0


**Reading.** Almost every relationship closes at 100 %. The losses are concentrated in three places:

| Relationship | What is missing | Why it matters |
|---|---|---|
| `BSEG.KUNNR` → `KNA1` | Customer `0000100999` (3 lines) | Receivables with no customer master: an inner join drops them from competency question 1 |
| `BSEG` (`BUKRS`+`HKONT`) → `SKB1` | Account `0000800100` in company code 3000 (2 lines) | Revenue account without company-code configuration |
| `SKAT.SAKNR` → `SKA1` | Account `0000455000` (text in EN and ES) | Text for an account that does not exist in the chart |

### 3.1 Unmatched rows in detail
Every row that found no parent, grouped by relationship. This is the evidence behind the log entries below.

In [9]:
def get_orphans(child, parent, ck, pk=None):
    pk = pk or ck
    C = tables[child]
    P = tables[parent][pk].drop_duplicates().set_axis(ck, axis=1)
    with_key = C[(C[ck] != "").all(axis=1)]
    m = with_key.merge(P, on=ck, how="left", indicator=True)
    return m[m["_merge"] == "left_only"].drop(columns="_merge").reset_index(drop=True)


orphan_rows = {}
for r in RELATIONSHIPS:
    child, parent, ck = r[0], r[1], r[2]
    pk = r[3] if len(r) > 3 else None
    o = get_orphans(child, parent, ck, pk)
    if len(o):
        orphan_rows[f"{child} -> {parent} ({'+'.join(ck)})"] = o

for name, o in orphan_rows.items():
    print(f"\n{name}: {len(o)} rows")
    display(o)




BSEG -> KNA1 (KUNNR): 3 rows


,BUKRS,BELNR,GJAHR,BUZEI,BSCHL,KOART,SHKZG,HKONT,KUNNR,LIFNR,WRBTR,DMBTR,MWSKZ,KOSTL,ZTERM,ZFBDT,AUGBL,AUGDT,SGTXT
0,1000,1800000245,2025,001,01,D,S,0000140000,0000100999,,6265.96,6265.96,,,NT30,20251011,,,Sales invoice
1,1000,1800000251,2025,001,01,D,S,0000140000,0000100999,,4718.3,4718.3,,,NT30,20251015,,,Sales invoice
2,1000,1800000281,2025,001,01,D,S,0000140000,0000100999,,5468.66,5468.66,,,NT30,20251101,,,Sales invoice



BSEG -> SKB1 (BUKRS+HKONT): 2 rows


,BUKRS,BELNR,GJAHR,BUZEI,BSCHL,KOART,SHKZG,HKONT,KUNNR,LIFNR,WRBTR,DMBTR,MWSKZ,KOSTL,ZTERM,ZFBDT,AUGBL,AUGDT,SGTXT
0,3000,1800000017,2025,002,50,S,H,0000800100,,,1793.71,1793.71,A2,,,,,,
1,3000,1800000052,2025,002,50,S,H,0000800100,,,5291.71,5291.71,A2,,,,,,



SKAT -> SKA1 (KTOPL+SAKNR): 2 rows


,SPRAS,KTOPL,SAKNR,TXT20,TXT50
0,E,NVCA,0000455000,Impairment losses,Impairment losses on assets
1,S,NVCA,0000455000,Deterioro de valor,Pérdidas por deterioro de activos


## 4 · Looking at the tables
First look at each table, one by one, before writing any rule: what it describes, how big it is and what the rows look like. Whatever looks odd here becomes a check and, if confirmed, a log entry.

In [10]:
def show(name):
    d = tables[name]
    desc = dictionary.loc[dictionary["table"] == name, "table_description"].iloc[0]
    print(f"{name} - {desc} | {len(d)} rows x {d.shape[1]} columns")
    return d

### 4.1 T001 · Company codes

In [11]:
show("T001")

T001 - Company codes | 3 rows x 8 columns


,BUKRS,BUTXT,ORT01,LAND1,WAERS,SPRAS,KTOPL,PERIV
0,1000,Nordvia Retail Iberia S.L.,Barcelona,ES,EUR,S,NVCA,K4
1,2000,Nordvia Retail US Inc.,Chicago,US,USD,E,NVCA,K4
2,3000,Nordvia Retail UK Ltd.,Manchester,GB,GBP,E,NVCA,K4


**Visual check.** 3 rows: company codes `1000` (ES, EUR), `2000` (US, USD) and `3000` (GB, GBP). All use chart of accounts `NVCA` and fiscal year variant `K4` (calendar year), and `SPRAS` is `S` or `E`, as the README says. **No issues.**

### 4.2 SKA1 · G/L accounts (chart of accounts)

In [11]:
show("SKA1")

SKA1 - G/L accounts (chart of accounts) | 27 rows x 7 columns


,KTOPL,SAKNR,KTOKS,XBILK,GVTYP,ERDAT,ERNAM
0,NVCA,0000100000,SAKO,X,,20190315,MIGRATION
1,NVCA,0000113100,SAKO,X,,20190315,MIGRATION
2,NVCA,0000140000,SAKO,X,,20190315,MIGRATION
3,NVCA,0000154000,SAKO,X,,20190315,MIGRATION
4,NVCA,0000160000,SAKO,X,,20190315,MIGRATION
5,NVCA,0000175000,SAKO,X,,20190315,MIGRATION
6,NVCA,0000200000,SAKO,X,,20190315,MIGRATION
7,NVCA,0000210000,SAKO,X,,20190315,MIGRATION
8,NVCA,0000230000,SAKO,X,,20190315,MIGRATION
9,NVCA,0000290000,SAKO,X,,20190315,MIGRATION


**Visual check.** 27 accounts, all in chart `NVCA`. Each one is flagged either balance sheet (`XBILK`: 10 accounts, 1xxxxx-2xxxxx) or P&L (`GVTYP`: 17 accounts, 4xxxxx-8xxxxx), never both and never neither. All were created on 2019-03-15 by `MIGRATION`, so the creation date carries no information (not an error). **No issues.**

### 4.3 SKAT · G/L account texts

In [13]:
show("SKAT")

SKAT - G/L account texts | 55 rows x 5 columns


,SPRAS,KTOPL,SAKNR,TXT20,TXT50
0,E,NVCA,0000100000,Petty cash,Petty cash
1,S,NVCA,0000100000,Caja,Caja
2,E,NVCA,0000113100,Bank - main account,Bank - main account
3,S,NVCA,0000113100,Banco - cuenta princ,Banco - cuenta principal
4,E,NVCA,0000140000,Trade receivables,Trade receivables
5,S,NVCA,0000140000,Clientes,Clientes
6,E,NVCA,0000154000,Input VAT,Input VAT
7,S,NVCA,0000154000,IVA soportado,IVA soportado
8,E,NVCA,0000160000,Trade payables,Trade payables
9,S,NVCA,0000160000,Proveedores,Proveedores


In [13]:
skat = tables["SKAT"]

# Accounts that do not have their text in both languages
langs = skat.groupby("SAKNR")["SPRAS"].apply(lambda s: ",".join(sorted(s)))
display(skat[skat["SAKNR"].isin(langs[langs != "E,S"].index)])

# README: short text up to 20 characters, long text up to 50, languages E or S
print("longest TXT20:", skat["TXT20"].str.len().max(), "| longest TXT50:", skat["TXT50"].str.len().max())
print("languages:", skat["SPRAS"].unique())

,SPRAS,KTOPL,SAKNR,TXT20,TXT50
48,E,NVCA,0000499000,Miscellaneous expens,Miscellaneous expense


longest TXT20: 20 | longest TXT50: 33
languages: <StringArray>
['E', 'S']
Length: 2, dtype: str


**Findings.**
- Account `0000499000` (Miscellaneous expense) only has an English text. Every other account has both languages. *Low severity: Spanish labels will be missing for one account.*
- Text lengths and languages follow the README.
- Account `0000455000` has texts but no account in `SKA1` (already seen in 3.1).

### 4.4 SKB1 · G/L accounts (company code)

In [15]:
show("SKB1")

SKB1 - G/L accounts (company code) | 80 rows x 7 columns


,BUKRS,SAKNR,WAERS,MITKZ,XOPVW,FSTAG,XKRES
0,1000,0000100000,EUR,,,G001,X
1,2000,0000100000,USD,,,G001,X
2,3000,0000100000,GBP,,,G001,X
3,1000,0000113100,EUR,,,G001,X
4,2000,0000113100,USD,,,G001,X
5,3000,0000113100,GBP,,,G001,X
6,1000,0000140000,EUR,D,X,G001,X
7,2000,0000140000,USD,D,X,G001,X
8,3000,0000140000,GBP,D,X,G001,X
9,1000,0000154000,EUR,,,G001,X


In [15]:
skb1 = tables["SKB1"]

# Accounts of the chart that are missing in each company code
by_company = skb1.groupby("BUKRS")["SAKNR"].apply(set)
all_accounts = set(tables["SKA1"]["SAKNR"])
print({co: sorted(all_accounts - accs) for co, accs in by_company.items()})

# Account currency different from the company currency
m = skb1.merge(tables["T001"][["BUKRS", "WAERS"]], on="BUKRS", suffixes=("_account", "_company"))
display(m[m["WAERS_account"] != m["WAERS_company"]])

# README values: reconciliation type, open items, field status group, line-item display
for col in ["MITKZ", "XOPVW", "FSTAG", "XKRES"]:
    print(col, skb1[col].value_counts().to_dict())

{'1000': [], '2000': [], '3000': ['0000800100']}


,BUKRS,SAKNR,WAERS_account,MITKZ,XOPVW,FSTAG,XKRES,WAERS_company


MITKZ {'': 74, 'D': 3, 'K': 3}
XOPVW {'': 74, 'X': 6}
FSTAG {'G001': 39, 'G004': 36, 'G029': 5}
XKRES {'X': 80}


**Findings.**
- Company code `3000` is missing account `0000800100`, which exists in `1000` and `2000`. This is the origin of the 2 `BSEG` lines without a parent seen in 3.1. *Medium severity: a revenue account without configuration in one company code.*
- Account currencies always match the company currency.
- `MITKZ` (3 `D`, 3 `K`), `XOPVW`, `FSTAG` (`G001`, `G004`, `G029`) and `XKRES` only hold values described in the README.

### 4.5 CSKS · Cost centers

In [19]:
show("CSKS")

CSKS - Cost centers | 25 rows x 9 columns


,KOKRS,KOSTL,DATBI,DATAB,BUKRS,KOSAR,VERAK,KTEXT,LTEXT
0,NV00,0000011010,99991231,20190101,1000,V,M. Puig,Administration,Administration
1,NV00,0000011020,99991231,20190101,1000,V,L. Romero,Finance,Finance
2,NV00,0000011030,99991231,20190101,1000,H,A. Serra,IT,IT
3,NV00,0000011040,99991231,20190101,1000,W,C. Vidal,Marketing,Marketing
4,NV00,0000011050,99991231,20190101,1000,W,J. Ortega,Sales,Sales
5,NV00,0000011060,99991231,20190101,1000,H,P. Navarro,Logistics,Logistics
6,NV00,0000011070,99991231,20190101,1000,W,R. Font,Store Barcelona Diag,Store Barcelona Diagonal
7,NV00,0000011080,20250630,20190101,1000,W,S. Gil,Store Valencia Centr,Store Valencia Centro
8,NV00,0000021010,99991231,20190101,2000,V,K. Miller,Administration,Administration
9,NV00,0000021020,99991231,20190101,2000,V,D. Brown,Finance,Finance


**Visual check.** 25 cost centers of controlling area `NV00`. All have an owner and a category in `V`, `H` or `W`, as the README says. 24 are valid until `99991231` (no end date). One exception: `0000011080` (Store Valencia Centro) is valid only until 2025-06-30, and `0000011090` (Store Valencia Ruzafa) starts on 2025-07-01. Whether postings kept using the old one is checked in 4.11. **No issues in the table itself.**

### 4.6 T003 · Document types

In [20]:
show("T003")

T003 - Document types | 7 rows x 3 columns


,BLART,LTEXT,NUMKR
0,DR,Customer invoice,18
1,DG,Customer credit memo,16
2,DZ,Customer payment,14
3,KR,Vendor invoice,19
4,KZ,Vendor payment,15
5,SA,G/L account document,01
6,AB,Accounting document (reversals),02


In [18]:
t003 = tables["T003"]
usage = tables["BKPF"]["BLART"].value_counts()
t003.assign(documents=t003["BLART"].map(usage).fillna(0).astype(int))

,BLART,LTEXT,NUMKR,documents
0,DR,Customer invoice,18,683
1,DG,Customer credit memo,16,22
2,DZ,Customer payment,14,590
3,KR,Vendor invoice,19,481
4,KZ,Vendor payment,15,422
5,SA,G/L account document,01,148
6,AB,Accounting document (reversals),02,7


**Findings.** All 7 document types are used, and each has its own number range. `AB` (reversals) is used by only 7 documents. **No issues.**

### 4.7 TBSL · Posting keys

In [22]:
show("TBSL")

TBSL - Posting keys | 8 rows x 4 columns


,BSCHL,SHKZG,KOART,LTEXT
0,01,S,D,Invoice
1,11,H,D,Credit memo
2,15,H,D,Incoming payment
3,21,S,K,Credit memo
4,25,S,K,Outgoing payment
5,31,H,K,Invoice
6,40,S,S,Debit entry
7,50,H,S,Credit entry


In [20]:
tbsl = tables["TBSL"]
bseg = tables["BSEG"]

# Lines whose SHKZG / KOART differ from what their posting key defines
m = bseg.merge(tbsl[["BSCHL", "SHKZG", "KOART"]], on="BSCHL", suffixes=("_line", "_key"))
print("inconsistent lines:", ((m["SHKZG_line"] != m["SHKZG_key"]) | (m["KOART_line"] != m["KOART_key"])).sum())

# Lines per posting key
tbsl.assign(lines=tbsl["BSCHL"].map(bseg["BSCHL"].value_counts()).fillna(0).astype(int))

inconsistent lines: 0


,BSCHL,SHKZG,KOART,LTEXT,lines
0,01,S,D,Invoice,683
1,11,H,D,Credit memo,26
2,15,H,D,Incoming payment,590
3,21,S,K,Credit memo,3
4,25,S,K,Outgoing payment,422
5,31,H,K,Invoice,481
6,40,S,S,Debit entry,2230
7,50,H,S,Credit entry,1627


**Findings.** All 8 posting keys are used, and no `BSEG` line contradicts the debit/credit indicator or account type of its key. `SHKZG` and `KOART` only hold the values in the README. **No issues.**

### 4.8 KNA1 · Customers

In [23]:
show("KNA1")

KNA1 - Customers | 156 rows x 10 columns


,KUNNR,NAME1,LAND1,ORT01,PSTLZ,STCD1,KTOKD,ERDAT,LOEVM,ZTERM
0,0000100001,Silver Comercial S.A.,ES,Bilbao,48001,A28420415,KUNA,20210629,,0001
1,0000100002,Iber Retail S.L.,ES,Barcelona,08001,B70225155,KUNA,20231101,,NT30
2,0000100003,Sierra Retail S.L.U.,ES,Bilbao,48001,B27613013,KUNA,20220501,,0001
3,0000100004,Meridian Distribuciones S.L.,ES,Barcelona,08001,A96026163,KUNA,20200226,,NT30
4,0000100005,Terra Distribuciones S.A.,ES,Bilbao,48001,B94019938,KUNA,20180830,,NT60
5,0000100006,Sol Brands S.L.U.,ES,Zaragoza,50001,B36572987,KUNA,20230515,,NT60
6,0000100007,Evergreen Retail S.L.U.,ES,Barcelona,08001,A44018583,KUNA,20230723,,0001
7,0000100008,Maple Trading S.A.,ES,Barcelona,08001,A61065291,KUNA,20181127,,NT30
8,0000100009,Nova Distribuciones S.L.U.,ES,Sevilla,41001,A73537199,KUNA,20241031,,NT60
9,0000100010,Northstar Retail S.L.U.,ES,Zaragoza,50001,B96333984,KUNA,20210513,,NT30


In [22]:
kna1 = tables["KNA1"]

# README conventions: values, identifier length and dates
for col in ["KTOKD", "ZTERM", "LOEVM"]:
    print(col, kna1[col].value_counts().to_dict())
print("KUNNR lengths:", kna1["KUNNR"].str.len().unique())
print("invalid ERDAT:", pd.to_datetime(kna1["ERDAT"], format="%Y%m%d", errors="coerce").isna().sum())

# Account group against country
display(pd.crosstab(kna1["KTOKD"], kna1["LAND1"]))

# Country that is not two uppercase letters
display(kna1[~kna1["LAND1"].str.fullmatch(r"[A-Z]{2}")])

# Missing Tax ID, by account group
display(kna1[kna1["STCD1"] == ""].groupby("KTOKD").size())

# Repeated Tax ID or name
display(kna1[(kna1["STCD1"] != "") & kna1.duplicated("STCD1", keep=False)])
display(kna1[kna1.duplicated("NAME1", keep=False)].sort_values("NAME1"))

# Customers flagged as deleted, and how many BSEG lines they have
deleted = kna1[kna1["LOEVM"] != ""]
deleted.assign(bseg_lines=deleted["KUNNR"].map(tables["BSEG"]["KUNNR"].value_counts()).fillna(0).astype(int))

KTOKD {'KUNA': 128, 'KUNX': 25, 'CPD': 3}
ZTERM {'NT30': 57, '0001': 41, 'NT60': 33, 'NT15': 25}
LOEVM {'': 155, 'X': 1}
KUNNR lengths: [10]
invalid ERDAT: 0


LAND1,DE,ES,FR,GB,IT,PT,US,es
KTOKD,,,,,,,,
CPD,0,1,0,1,0,0,1,0
KUNA,0,56,0,28,0,0,42,2
KUNX,5,0,7,2,2,4,5,0


,KUNNR,NAME1,LAND1,ORT01,PSTLZ,STCD1,KTOKD,ERDAT,LOEVM,ZTERM
16,0000100017,Coral Retail S.A.,es,Madrid,28001,A16525001,KUNA,20180514,,NT15
40,0000100041,Orion Distribuciones S.A.,es,Barcelona,08001,B83650017,KUNA,20240410,,NT15


KTOKD
CPD    3
dtype: int64

,KUNNR,NAME1,LAND1,ORT01,PSTLZ,STCD1,KTOKD,ERDAT,LOEVM,ZTERM


,KUNNR,NAME1,LAND1,ORT01,PSTLZ,STCD1,KTOKD,ERDAT,LOEVM,ZTERM
153,0000100154,One-time customer,ES,,,,CPD,20190101,,0001
154,0000100155,One-time customer,US,,,,CPD,20190101,,0001
155,0000100156,One-time customer,GB,,,,CPD,20190101,,0001


,KUNNR,NAME1,LAND1,ORT01,PSTLZ,STCD1,KTOKD,ERDAT,LOEVM,ZTERM,bseg_lines
22,0000100023,Harbor Comercial S.A.,ES,Bilbao,48001,A88133160,KUNA,20240524,X,0001,2


**Findings.**
- Account groups, payment terms and deletion flag only hold values from the README; identifiers have 10 digits and dates are valid.
- **Defect:** customers `0000100017` and `0000100041` have the country in lowercase (`es`), so the same country would become two different values. *Low severity: normalize to uppercase.*
- **Normal:** 3 one-time customers (`CPD`) share the name `One-time customer` and have no Tax ID, city or postal code; this is how this account group works. *Do not merge them.*
- **Normal:** no Tax ID is shared by two customers.
- **Normal:** `KUNA` is not only Spain (58 customers in `ES`, 28 in `GB`, 42 in `US`), but "domestic" is relative to the selling company. Compared against the country of the company code in `BSEG`/`T001`, every `KUNA` line is domestic and every `KUNX` line is foreign (see 5.7).
- **Normal:** customer `0000100023` is flagged for deletion (`LOEVM`) and has 2 settled `BSEG` lines. The flag does not block postings and the history is real; keep it and expose the flag.

### 4.9 LFA1 · Vendors

In [26]:
show("LFA1")

LFA1 - Vendors | 72 rows x 10 columns


,LIFNR,NAME1,LAND1,ORT01,PSTLZ,STCD1,KTOKK,ERDAT,SPERR,ZTERM
0,0000200001,Granite Textiles S.L.U.,ES,Sevilla,41001,A38560217,LIEF,20180824,,NT45
1,0000200002,Summit Foods S.L.U.,ES,Valencia,46001,B21867292,LIEF,20210906,,NT60
2,0000200003,Zenith Foods S.A.,ES,Sevilla,41001,A95179805,LIEF,20220712,,NT45
3,0000200004,Harbor Systems S.L.U.,ES,Bilbao,48001,A32661340,LIEF,20190819,,NT45
4,0000200005,Zenith Energy S.L.,ES,Bilbao,48001,A85090550,LIEF,20240804,,NT60
5,0000200006,Maple Supplies S.L.,ES,Sevilla,41001,B99688337,LIEF,20240407,,NT60
6,0000200007,Riverside Textiles S.A.,ES,Málaga,29001,A82517373,LIEF,20190517,,NT60
7,0000200008,Orion Media S.L.,ES,Zaragoza,50001,A69663290,LIEF,20200815,,NT60
8,0000200009,Nova Foods S.L.,ES,Zaragoza,50001,A83705832,LIEF,20211127,,NT60
9,0000200010,Crest Textiles S.L.U.,ES,Sevilla,41001,A96517240,LIEF,20191017,,NT60


In [24]:
lfa1 = tables["LFA1"]

# README conventions: values, identifier length and dates
for col in ["KTOKK", "ZTERM", "SPERR"]:
    print(col, lfa1[col].value_counts().to_dict())
print("LIFNR lengths:", lfa1["LIFNR"].str.len().unique())
print("invalid ERDAT:", pd.to_datetime(lfa1["ERDAT"], format="%Y%m%d", errors="coerce").isna().sum())

# Country that is not two uppercase letters
display(lfa1[~lfa1["LAND1"].str.fullmatch(r"[A-Z]{2}")])

# Missing Tax ID, by account group
display(lfa1[lfa1["STCD1"] == ""].groupby("KTOKK").size())

# Same Tax ID, or same name once case, punctuation and spaces are ignored
norm = lfa1["NAME1"].str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
display(lfa1[(lfa1["STCD1"] != "") & lfa1.duplicated("STCD1", keep=False)].sort_values("STCD1"))
display(lfa1[norm.duplicated(keep=False)].sort_values("NAME1"))

# BSEG lines of the repeated vendors
tables["BSEG"][tables["BSEG"]["LIFNR"].isin(["0000200004", "0000200072"])].groupby("LIFNR").size()

KTOKK {'LIEF': 66, 'EMPL': 6}
ZTERM {'NT60': 32, 'NT45': 23, 'NT30': 11, '0001': 6}
SPERR {'': 72}
LIFNR lengths: [10]
invalid ERDAT: 0


,LIFNR,NAME1,LAND1,ORT01,PSTLZ,STCD1,KTOKK,ERDAT,SPERR,ZTERM


KTOKK
EMPL    6
dtype: int64

,LIFNR,NAME1,LAND1,ORT01,PSTLZ,STCD1,KTOKK,ERDAT,SPERR,ZTERM
3,0000200004,Harbor Systems S.L.U.,ES,Bilbao,48001,A32661340,LIEF,20190819,,NT45
71,0000200072,HARBOR SYSTEMS SLU,ES,Bilbao,48001,A32661340,LIEF,20250312,,NT45


,LIFNR,NAME1,LAND1,ORT01,PSTLZ,STCD1,KTOKK,ERDAT,SPERR,ZTERM
71,0000200072,HARBOR SYSTEMS SLU,ES,Bilbao,48001,A32661340,LIEF,20250312,,NT45
3,0000200004,Harbor Systems S.L.U.,ES,Bilbao,48001,A32661340,LIEF,20190819,,NT45


LIFNR
0000200004    10
0000200072     6
dtype: int64

**Findings.**
- Account groups, payment terms and the posting block only hold values from the README (no vendor is blocked); identifiers have 10 digits, dates are valid and countries are well formed.
- The 6 vendors without Tax ID are all employees (`EMPL`), which is expected.
- Vendors `0000200004` (*Harbor Systems S.L.U.*, created 2019) and `0000200072` (*HARBOR SYSTEMS SLU*, created 2025) share Tax ID `A32661340`, city and postal code; the names only differ in case and punctuation. They are the same company, with 10 and 6 `BSEG` lines respectively, so the spend is split in two. *Medium severity: unify by Tax ID.*

### 4.10 BKPF · Document header

In [19]:
show("BKPF")

BKPF - Document header | 2353 rows x 15 columns


,BUKRS,BELNR,GJAHR,BLART,BLDAT,BUDAT,MONAT,CPUDT,USNAM,WAERS,KURSF,XBLNR,BKTXT,STBLG,STJAH
0,1000,0100000001,2025,SA,20250101,20250101,01,20250101,MIGRATION,EUR,1.00000,,Opening balances 2025,,
1,1000,0100000002,2025,SA,20250131,20250131,01,20250131,BATCH_HR,EUR,1.00000,,Payroll 01/2025,,
2,1000,0100000003,2025,SA,20250131,20250131,01,20250131,BATCH_HR,EUR,1.00000,,Payroll payout 01/2025,,
3,1000,0100000004,2025,SA,20250131,20250131,01,20250131,BATCH_AA,EUR,1.00000,,Depreciation 01/2025,,
4,1000,0100000005,2025,SA,20250131,20250131,01,20250131,MLOPEZ,EUR,1.00000,,Bank charges 01/2025,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2348,3000,1900000096,2025,KR,20251214,20251219,12,20251219,RTAYLOR,GBP,1.00000,F2025752/21,,,
2349,3000,1900000097,2025,KR,20251217,20251221,12,20251221,RTAYLOR,GBP,1.00000,F2025727/450,,,
2350,3000,1900000098,2025,KR,20251215,20251225,12,20251225,AJONES,GBP,1.00000,F2025083/955,,,
2351,3000,1900000099,2025,KR,20251226,20251226,12,20251226,AJONES,GBP,1.00000,F2025273/392,,,


In [26]:
bkpf = tables["BKPF"]

# README conventions: dates, document number, exchange rate
for col in ["BLDAT", "BUDAT", "CPUDT"]:
    d = pd.to_datetime(bkpf[col], format="%Y%m%d", errors="coerce")
    print(col, "| invalid:", d.isna().sum(), "| from", d.min().date(), "to", d.max().date())
print("BELNR lengths:", bkpf["BELNR"].str.len().unique())
print("KURSF not a positive number:", (~(pd.to_numeric(bkpf["KURSF"], errors="coerce") > 0)).sum())

# Fiscal year / period agree with the posting date; date order
print("year/period different from posting date:",
      ((bkpf["GJAHR"] != bkpf["BUDAT"].str[:4]) | (bkpf["MONAT"] != bkpf["BUDAT"].str[4:6])).sum())
print("BLDAT after BUDAT:", (bkpf["BLDAT"] > bkpf["BUDAT"]).sum(), "| CPUDT before BUDAT:", (bkpf["CPUDT"] < bkpf["BUDAT"]).sum())

# Documents that start before the fiscal year, or are entered after it ends
print("BLDAT before 2025:", (bkpf["BLDAT"] < "20250101").sum(), "| CPUDT after 2025-12-31:", (bkpf["CPUDT"] > "20251231").sum())
display(bkpf[bkpf["CPUDT"] > "20251231"][["BUKRS", "BELNR", "BLART", "BUDAT", "CPUDT"]])

# Exchange rate against the company currency
co = bkpf.merge(tables["T001"][["BUKRS", "WAERS"]], on="BUKRS", suffixes=("", "_company"))
print("same currency but rate <> 1:", ((co["WAERS"] == co["WAERS_company"]) & (co["KURSF"].astype(float) != 1)).sum())
print("foreign currency but rate = 1:", ((co["WAERS"] != co["WAERS_company"]) & (co["KURSF"].astype(float) == 1)).sum())
print("foreign-currency documents:", (co["WAERS"] != co["WAERS_company"]).sum())

# Repeated reference within company code and document type
refs = bkpf[(bkpf["XBLNR"] != "") & bkpf.duplicated(["BUKRS", "BLART", "XBLNR"], keep=False)]
display(refs.sort_values(["BUKRS", "BLART", "XBLNR"]))

# Reversal documents (STBLG filled) and their counterpart
display(bkpf[bkpf["STBLG"] != ""])

# Opening balances: entered by the migration user
bkpf[bkpf["USNAM"] == "MIGRATION"]

BLDAT | invalid: 0 | from 2024-12-25 to 2025-12-31
BUDAT | invalid: 0 | from 2025-01-01 to 2025-12-31
CPUDT | invalid: 0 | from 2025-01-01 to 2026-01-02
BELNR lengths: [10]
KURSF not a positive number: 0
year/period different from posting date: 0
BLDAT after BUDAT: 0 | CPUDT before BUDAT: 0
BLDAT before 2025: 8 | CPUDT after 2025-12-31: 13


,BUKRS,BELNR,BLART,BUDAT,CPUDT
45,1000,0100000046,SA,20251231,20260102
46,1000,0100000047,SA,20251231,20260102
47,1000,0100000048,SA,20251231,20260102
334,1000,1400000283,DZ,20251231,20260102
1154,2000,0100000046,SA,20251231,20260101
1156,2000,0100000048,SA,20251231,20260101
1157,2000,0100000049,SA,20251231,20260101
1344,2000,1400000186,DZ,20251231,20260101
1705,2000,1800000220,DR,20251231,20260102
1854,2000,1900000149,KR,20251231,20260101


same currency but rate <> 1: 0
foreign currency but rate = 1: 0
foreign-currency documents: 76


,BUKRS,BELNR,GJAHR,BLART,BLDAT,BUDAT,MONAT,CPUDT,USNAM,WAERS,KURSF,XBLNR,BKTXT,STBLG,STJAH
960,1000,1900000083,2025,KR,20250515,20250515,05,20250515,JGARCIA,EUR,1.00000,F2025966/750,,,
1108,1000,1900000231,2025,KR,20250515,20250519,05,20250519,APUJOL,EUR,1.00000,F2025966/750,,,
1499,2000,1800000014,2025,DR,20250123,20250123,01,20250123,KWONG,USD,1.00000,225-11649,,,
1626,2000,1800000141,2025,DR,20250810,20250810,08,20250810,KWONG,USD,1.00000,225-11649,,,


,BUKRS,BELNR,GJAHR,BLART,BLDAT,BUDAT,MONAT,CPUDT,USNAM,WAERS,KURSF,XBLNR,BKTXT,STBLG,STJAH
50,1000,0200000001,2025,AB,20250217,20250217,02,20250218,MLOPEZ,EUR,1.00000,,Reversal - entry error,1900000026,2025
51,1000,0200000002,2025,AB,20250822,20250822,08,20250822,MLOPEZ,EUR,1.00000,,Reversal - entry error,1800000187,2025
731,1000,1800000187,2025,DR,20250819,20250819,08,20250820,APUJOL,EUR,1.00000,125-61992,,0200000002,2025
903,1000,1900000026,2025,KR,20250205,20250211,02,20250211,MLOPEZ,EUR,1.00000,F2025073/666,,0200000001,2025
1158,2000,0200000001,2025,AB,20250203,20250203,02,20250204,JSMITH,USD,1.00000,,Reversal - entry error,1800000019,2025
1504,2000,1800000019,2025,DR,20250128,20250128,01,20250128,JSMITH,USD,1.00000,225-47809,,0200000001,2025
1905,3000,0200000001,2025,AB,20251231,20251231,12,20260101,RTAYLOR,GBP,1.00000,,Reversal - entry error,1800000129,2025
1906,3000,0200000002,2025,AB,20250701,20250701,07,20250702,RTAYLOR,GBP,1.00000,,Reversal - entry error,1900000041,2025
1907,3000,0200000003,2025,AB,20250408,20250408,04,20250409,RTAYLOR,GBP,1.00000,,Reversal - entry error,1800000042,2025
1908,3000,0200000004,2025,AB,20251129,20251129,11,20251129,AJONES,GBP,1.00000,,Reversal - entry error,1900000093,2025


,BUKRS,BELNR,GJAHR,BLART,BLDAT,BUDAT,MONAT,CPUDT,USNAM,WAERS,KURSF,XBLNR,BKTXT,STBLG,STJAH
0,1000,0100000001,2025,SA,20250101,20250101,01,20250101,MIGRATION,EUR,1.00000,,Opening balances 2025,,
1109,2000,0100000001,2025,SA,20250101,20250101,01,20250101,MIGRATION,USD,1.00000,,Opening balances 2025,,
1856,3000,0100000001,2025,SA,20250101,20250101,01,20250101,MIGRATION,GBP,1.00000,,Opening balances 2025,,


**Findings.**
- Dates are valid `YYYYMMDD` and posting dates all fall in 2025; fiscal year and period always agree with the posting date; the document date is never after the posting date; exchange rates are positive numbers.
- **Normal:** 8 documents have a document date (`BLDAT`) in December 2024 but are posted in 2025 (invoices issued before the year start). Aging starts from `ZFBDT`.
- **Normal:** 13 documents posted on 2025-12-31 were entered on 2026-01-01 or 2026-01-02 (year-end entries). The reference date is `BUDAT`.
- Exchange rate is 1 whenever the document is in the company currency, and differs from 1 in the 76 foreign-currency documents.
- **Defect:** `KR` `F2025966/750` in company `1000` (documents `1900000083` and `1900000231`) was entered twice: same vendor, invoice number, amount and invoice date, neither reversed, and each one cleared by its own payment (`1500000094`, `1500000095`), so it was paid twice. *Medium severity: mark as a probable duplicate.*
- **Normal:** `DR` `225-11649` in company `2000` (documents `1800000014` and `1800000141`) is a reference collision between different customers and amounts; `XBLNR` just cannot be used as an identifier.
- **Condition:** 14 documents are involved in reversals: 7 reversal documents (`AB`) and the 7 originals that point to them through `STBLG`; 2 of the reversals were posted in the period after their original. *High: exclude both from balances and P&L, and model `reversedBy`.*
- **Condition:** 3 `SA` documents entered by `MIGRATION` on 2025-01-01 are the opening balances (one per company code): bank, fixed assets and retained earnings only, so no receivable or payable is brought forward. *High for balance-sheet figures: treat apart from the year's activity.*
- **Condition:** there is no exchange-rate table (the challenge says so and asks to derive the rates from the data). The only rates are the `KURSF` of the 76 foreign-currency documents: USD/EUR has no document in January and GBP/EUR none in August (both can be crossed through the other currency), and the rate moves by up to 0.7 % within a month. *High for any figure in euros: build a monthly rate table and state the method.*

### 4.11 BSEG · Document line items

In [29]:
show("BSEG")

BSEG - Document line items | 6062 rows x 19 columns


,BUKRS,BELNR,GJAHR,BUZEI,BSCHL,KOART,SHKZG,HKONT,KUNNR,LIFNR,WRBTR,DMBTR,MWSKZ,KOSTL,ZTERM,ZFBDT,AUGBL,AUGDT,SGTXT
0,1000,0100000001,2025,001,40,S,S,0000113100,,,650000.0,650000.0,,,,,,,
1,1000,0100000001,2025,002,40,S,S,0000200000,,,180000.0,180000.0,,,,,,,
2,1000,0100000001,2025,003,50,S,H,0000290000,,,830000.0,830000.0,,,,,,,
3,1000,0100000002,2025,001,40,S,S,0000410000,,,6393.78,6393.78,,0000011010,,,,,
4,1000,0100000002,2025,002,40,S,S,0000411000,,,1918.13,1918.13,,0000011010,,,,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6057,3000,1900000099,2025,002,40,S,S,0000420000,,,4844.33,4844.33,V2,0000031080,,,,,rent
6058,3000,1900000099,2025,003,40,S,S,0000154000,,,968.87,968.87,V2,,,,,,
6059,3000,1900000100,2025,001,31,K,H,0000160000,,0000200060,5720.21,5720.21,,,NT60,20251222,,,
6060,3000,1900000100,2025,002,40,S,S,0000430000,,,4766.84,4766.84,V2,0000031030,,,,,it


In [28]:
bseg = tables["BSEG"]
bkpf = tables["BKPF"]
amount = pd.to_numeric(bseg["DMBTR"])             # converted only inside the query

# README conventions: values, identifier length and dates
for col in ["SHKZG", "KOART", "ZTERM", "MWSKZ"]:
    print(col, bseg[col].value_counts().to_dict())
for col in ["HKONT", "KUNNR", "LIFNR", "KOSTL", "BELNR", "AUGBL"]:
    print(col, "lengths:", bseg.loc[bseg[col] != "", col].str.len().unique())
for col in ["ZFBDT", "AUGDT"]:
    s = bseg.loc[bseg[col] != "", col]
    print(col, "| invalid:", pd.to_datetime(s, format="%Y%m%d", errors="coerce").isna().sum(), "| from", s.min(), "to", s.max())

# 1. Each document must balance: debits = credits (local currency)
signed = amount.where(bseg["SHKZG"] == "S", -amount)
balance = signed.groupby([bseg["BUKRS"], bseg["BELNR"], bseg["GJAHR"]]).sum().round(2)
print("unbalanced documents:", (balance != 0).sum())

# 2. Zero or negative amounts (README: amounts are always positive)
display(bseg[(amount <= 0) | (pd.to_numeric(bseg["WRBTR"]) <= 0)].groupby(["KOART", "HKONT"]).size())

# 3. Customer / vendor present exactly when the account type needs it
display(bseg[((bseg["KOART"] == "D") & (bseg["KUNNR"] == "")) | ((bseg["KOART"] != "D") & (bseg["KUNNR"] != ""))])
display(bseg[((bseg["KOART"] == "K") & (bseg["LIFNR"] == "")) | ((bseg["KOART"] != "K") & (bseg["LIFNR"] != ""))])

# 4. Local amount vs document amount x exchange rate (1 % tolerance)
m = bseg.merge(bkpf[["BUKRS", "BELNR", "GJAHR", "KURSF", "WAERS"]], on=["BUKRS", "BELNR", "GJAHR"])
m = m[pd.to_numeric(m["WRBTR"]) != 0]             # zero-amount lines are covered by check 2
expected = pd.to_numeric(m["WRBTR"]) * m["KURSF"].astype(float)
diff = m[(expected - pd.to_numeric(m["DMBTR"])).abs() > 0.01 * expected.abs()]
print("amount differs from rate:", len(diff))
display(diff.groupby(["WAERS", "KOART"]).size())

# 5. Expense lines (account group G004) without cost center
fstag = bseg.merge(tables["SKB1"][["BUKRS", "SAKNR", "FSTAG"]], left_on=["BUKRS", "HKONT"], right_on=["BUKRS", "SAKNR"])
display(fstag[(fstag["FSTAG"] == "G004") & (fstag["KOSTL"] == "")].groupby("HKONT").size())

# 6. Cost center used after its valid-to date
cc = bseg[bseg["KOSTL"] != ""].merge(bkpf[["BUKRS", "BELNR", "GJAHR", "BUDAT"]], on=["BUKRS", "BELNR", "GJAHR"])
cc = cc.merge(tables["CSKS"][["KOSTL", "DATBI"]], on="KOSTL")
display(cc[cc["BUDAT"] > cc["DATBI"]].groupby(["KOSTL", "DATBI"]).size())

# 7. Line payment terms (when informed) vs customer master
lines = bseg[(bseg["KUNNR"] != "") & (bseg["ZTERM"] != "")].merge(tables["KNA1"][["KUNNR", "ZTERM"]], on="KUNNR", suffixes=("_line", "_master"))
display(lines[lines["ZTERM_line"] != lines["ZTERM_master"]].groupby(["ZTERM_line", "ZTERM_master"]).size())

# 8. Due date (ZFBDT) before the document date (BLDAT)
d = bseg[bseg["ZFBDT"] != ""].merge(bkpf[["BUKRS", "BELNR", "GJAHR", "BLART", "BLDAT"]], on=["BUKRS", "BELNR", "GJAHR"])
display(d[d["ZFBDT"] < d["BLDAT"]].groupby("BLART").size())

# 9. Clearing: document and date must come together; clearing date not before posting; open items
cl = bseg[bseg["AUGDT"] != ""].merge(bkpf[["BUKRS", "BELNR", "GJAHR", "BUDAT"]], on=["BUKRS", "BELNR", "GJAHR"])
print("AUGBL without AUGDT:", ((bseg["AUGBL"] != "") & (bseg["AUGDT"] == "")).sum(),
      "| AUGDT without AUGBL:", ((bseg["AUGBL"] == "") & (bseg["AUGDT"] != "")).sum(),
      "| AUGDT before posting:", (cl["AUGDT"] < cl["BUDAT"]).sum())
print("open items (customer/vendor lines without clearing):", ((bseg["KOART"].isin(["D", "K"])) & (bseg["AUGBL"] == "")).sum())

# 10. Tax code agrees with the country of the company code (A1/V1 = ES, A2/V2 = UK)
tax = bseg.merge(tables["T001"][["BUKRS", "LAND1"]], on="BUKRS")
print("tax code from the wrong country:",
      ((tax["MWSKZ"].isin(["A1", "V1"]) & (tax["LAND1"] != "ES")) | (tax["MWSKZ"].isin(["A2", "V2"]) & (tax["LAND1"] != "GB"))).sum())

# 11. Line text: most frequent values
bseg["SGTXT"].value_counts().head(15)

SHKZG {'S': 3338, 'H': 2724}
KOART {'S': 3857, 'D': 1299, 'K': 906}
ZTERM {'': 4869, 'NT60': 378, 'NT30': 337, '0001': 290, 'NT45': 124, 'NT15': 64}
MWSKZ {'': 4198, 'A1': 468, 'V1': 428, 'A0': 351, 'A2': 248, 'V2': 198, 'V0': 171}
HKONT lengths: [10]


KUNNR lengths: [10]
LIFNR lengths: [10]
KOSTL lengths: [10]
BELNR lengths: [10]
AUGBL lengths: [10]
ZFBDT | invalid: 0 | from 20241225 to 20251231
AUGDT | invalid: 0 | from 20250101 to 20251231
unbalanced documents: 0


KOART  HKONT     
S      0000480000    24
       0000481000    12
dtype: int64

,BUKRS,BELNR,GJAHR,BUZEI,BSCHL,KOART,SHKZG,HKONT,KUNNR,LIFNR,WRBTR,DMBTR,MWSKZ,KOSTL,ZTERM,ZFBDT,AUGBL,AUGDT,SGTXT


,BUKRS,BELNR,GJAHR,BUZEI,BSCHL,KOART,SHKZG,HKONT,KUNNR,LIFNR,WRBTR,DMBTR,MWSKZ,KOSTL,ZTERM,ZFBDT,AUGBL,AUGDT,SGTXT


amount differs from rate: 13


... (3 outputs left out)

ZTERM_line  ZTERM_master
0001        NT30            4
            NT60            7
dtype: int64

BLART
AB    7
dtype: int64

AUGBL without AUGDT: 0 | AUGDT without AUGBL: 0 | AUGDT before posting: 0
open items (customer/vendor lines without clearing): 167
tax code from the wrong country: 0


SGTXT
                         3857
Sales invoice             687
Incoming payment          590
Outgoing payment          422
cogs                      198
mkt                        50
it                         49
rent                       47
cons                       41
trav                       36
off                        34
util                       29
Credit memo - returns      22
Name: count, dtype: int64

**Findings.**
- README conventions hold: values of `SHKZG`, `KOART`, `ZTERM` and `MWSKZ` are the documented ones; identifiers have 10 digits; dates are valid; tax codes match the country of the company code; amounts are never negative.
- All documents balance (debits = credits), and customer/vendor fields are filled exactly when the account type needs them.
- **Condition:** 36 lines have a document amount (`WRBTR`) of 0 and a local amount (`DMBTR`), in accounts `0000480000` (24) and `0000481000` (12): exchange-rate differences. The 36 customer/vendor payment lines (EUR, GBP and USD documents; 13 of them more than 1 % away) whose `DMBTR` is not `WRBTR` x `KURSF` all sit in documents that contain one of these lines, which carries the difference. This is how SAP clears at a different rate, not an error. *Medium: take the local amount from `DMBTR`, never recompute it.*
- **Defect:** 5 expense lines (`G004`, cost center required) have no cost center, in accounts `0000420000`, `0000430000`, `0000431000`, `0000440000` and `0000441000`. *Medium: cost center reports will be incomplete.*
- **Defect:** 4 lines were posted to cost center `0000011080` after it expired on 2025-06-30. Cost center `0000011090` (*Store Valencia Ruzafa*) starts the next day; that it replaces the old one is an assumption. *Low: keep, and flag the validity.*
- **Condition:** 11 customer lines have payment terms `0001` while the customer master says `NT30` (4) or `NT60` (7). All 11 are credit memos: the 22 credit memos always carry `0001`, and no invoice line differs from its master. *Low: read the agreed terms from the master.*
- **Condition:** the 22 customer credit memos are all open at the end of the year. Each one names its invoice in `XBLNR`; 19 of those invoices were already paid and 3 are still open. *Medium: net them in the customer's balance and keep them out of the age buckets.*
- **Normal:** 7 lines of reversal documents (`AB`) have a baseline date before the document date; they inherit it from the original.
- **Normal:** clearing is coherent (document and date always come together, no clearing before the posting). The 167 open items (89 customer invoices, 22 customer credit memos and 56 vendor invoices without clearing) are the receivables and payables at the end of the year.
- **Normal:** 484 lines have an abbreviated text (`cogs`, `mkt`, `it`, `rent`, `cons`, `trav`, `off`, `util`) and 3,857 have no text; the README defines no format for `SGTXT`.
- No amount outliers: amounts above 200,000 exist only in the opening-balance documents.

## 5 · Data quality log
Sections 3 and 4 produced observations; not every observation is a quality problem. Each one was checked against a **rule** (the README, the SAP configuration in the tables, referential integrity, or a stated business rule) and against the **rows that break it**. Only what breaks a rule, or can lead to wrong results if it is ignored, goes in the log, in two kinds:

- **Defect**: the data is wrong or inconsistent and could be corrected or reported to the source.
- **Condition to handle**: the data is correct as SAP produces it, but it has to be handled explicitly when balances, amounts or counts are calculated; ignoring it would give wrong results.

What was reviewed and found to be normal, with no effect, is listed in section 5.7, so the reasoning stays visible. Counts are computed from the data. Raw data is never modified; `impact_level` and `proposed_action` are proposals for the author to decide.

The log has one line per finding; **every** affected row is written to `data_quality_detail.csv`, one line per row, which repeats the rule, the reason, the impact and the proposed action of its finding so that the file can be read and filtered on its own.

### 5.1 Log structure

In [29]:
LOG_COLUMNS = ["id", "table", "column", "dimension", "rule_broken", "rule_source", "nature", "finding",
               "affected_rows", "total_rows", "pct_affected",
               "why_it_matters", "impact", "impact_level", "evidence",
               "proposed_action"]
log = pd.DataFrame(columns=LOG_COLUMNS)

detail = pd.DataFrame(columns=["id", "affected_table", "key_columns", "affected_key"])

def add_detail(issue_id, table, rows, cols):
    """Add EVERY affected row of one finding to the detail table, identified by its key columns."""
    global detail
    k = rows[cols].astype(str).agg("/".join, axis=1).drop_duplicates()
    detail = pd.concat([detail, pd.DataFrame({"id": issue_id, "affected_table": table,
                                              "key_columns": "/".join(cols), "affected_key": k.values})], ignore_index=True)

def log_issue(table, column, dimension, rule_broken, finding, n, total,
              why_it_matters, impact, impact_level, evidence, proposed_action,
              nature="Defect", rule_source="Referential integrity", affected=()):
    """Add one finding to the log. n / total are computed by the caller from the data."""
    global log
    row = {"id": f"DQ-{len(log) + 1:02d}", "table": table, "column": column, "dimension": dimension,
           "rule_broken": rule_broken, "rule_source": rule_source, "nature": nature,
           "finding": finding,
           "affected_rows": int(n), "total_rows": int(total),
           "pct_affected": round(100 * n / total, 2) if total else None,
           "why_it_matters": why_it_matters,
           "impact": impact, "impact_level": impact_level, "evidence": evidence,
           "proposed_action": proposed_action}
    log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    for table_name, rows, cols in affected:
        add_detail(row["id"], table_name, rows, cols)

reviewed = pd.DataFrame(columns=["table", "column", "observation", "rows", "why_it_is_normal"])

def accept(table, column, observation, n, why):
    """Record something that was reviewed and is NOT a quality problem."""
    global reviewed
    reviewed = pd.concat([reviewed, pd.DataFrame([{"table": table, "column": column, "observation": observation,
                                                   "rows": int(n), "why_it_is_normal": why}])], ignore_index=True)

def ymd(s):
    """20250630 -> 2025-06-30"""
    return f"{s[:4]}-{s[4:6]}-{s[6:]}"

bseg, bkpf = tables["BSEG"], tables["BKPF"]
BS_KEY = ["BUKRS", "BELNR", "GJAHR"]

### 5.2 Referential integrity

In [30]:
# --- Referential integrity (section 3) ---------------------------------------------
o = get_orphans("BSEG", "KNA1", ["KUNNR"])
log_issue("BSEG", "KUNNR", "Referential integrity",
    "Every customer used on a line must exist in the customer master (BSEG.KUNNR -> KNA1.KUNNR)",
    f"Customer {o['KUNNR'].iloc[0]} is used on {len(o)} lines but does not exist in KNA1.",
    len(o), (bseg["KUNNR"] != "").sum(),
    "Without the master record the customer has no name, country, account group or payment terms, and any inner join with KNA1 silently drops these lines.",
    "These 3 lines cannot be assigned to a known customer: balances per customer and their age would leave them out or show them under an unknown customer.",
    "High", "Section 3 and 3.1", "Keep the lines and flag them as orphan customer; do not drop them in joins.",
    affected=[("BSEG", o, BS_KEY + ["BUZEI"])])

o = get_orphans("BSEG", "SKB1", ["BUKRS", "HKONT"], ["BUKRS", "SAKNR"])
amount = pd.to_numeric(o["DMBTR"]).sum()
currency = tables["T001"].set_index("BUKRS").loc[o["BUKRS"].iloc[0], "WAERS"]
name = tables["SKAT"].query("SPRAS == 'E'").set_index("SAKNR").loc[o["HKONT"].iloc[0], "TXT50"]
log_issue("BSEG", "BUKRS + HKONT", "Referential integrity",
    "Every account posted in a company code must be configured for it (BSEG.BUKRS+HKONT -> SKB1.BUKRS+SAKNR)",
    f"Account {o['HKONT'].iloc[0]} ({name}) is posted {len(o)} times in company code {o['BUKRS'].iloc[0]} "
    f"({amount:,.2f} {currency}) but SKB1 has no record of it for that company code; it exists in the other two.",
    len(o), len(bseg),
    "SKB1 holds the company-level definition of the account (currency, field status, open-item management). Postings to an account that is not defined for the company are inconsistent with the configuration.",
    "Revenue of company 3000 sits on an account with no company-level definition: it would disappear from any report that joins postings with SKB1.",
    "Medium", "Section 3.1 and 4.4 (SKB1)", "Keep the lines; confirm with the business whether the account is valid for company code 3000.",
    affected=[("BSEG", o, BS_KEY + ["BUZEI"])])

o = get_orphans("SKAT", "SKA1", ["KTOPL", "SAKNR"])
used = (bseg["HKONT"] == o["SAKNR"].iloc[0]).sum()
log_issue("SKAT", "KTOPL + SAKNR", "Referential integrity",
    "Every account text must belong to an account of the chart (SKAT.KTOPL+SAKNR -> SKA1.KTOPL+SAKNR)",
    f"Account {o['SAKNR'].iloc[0]} has texts in English and Spanish ({o['TXT50'].iloc[0]}) but is not in SKA1; it is used on {used} BSEG lines.",
    len(o), len(tables["SKAT"]),
    "A text with no account is a label for something that does not exist in the chart of accounts.",
    "No effect on amounts, because the account is never posted. It would create a GLAccount without definition if loaded blindly.",
    "Low", "Section 3.1", "Do not load as GLAccount, or load it flagged as 'no definition'.",
    affected=[("SKAT", o, ["KTOPL", "SAKNR", "SPRAS"])])

headers = bkpf.merge(bseg[BS_KEY].drop_duplicates(), on=BS_KEY, how="left", indicator=True)
o = headers[headers["_merge"] == "left_only"]
log_issue("BKPF", "BELNR", "Referential integrity",
    "Every document header must have at least one line (BKPF -> BSEG)",
    f"Document {o['BELNR'].iloc[0]} ('{o['BKTXT'].iloc[0]}') has a header but no line items.",
    len(o), len(bkpf),
    "A posted document without lines carries no amount and cannot balance; it looks like an incomplete entry.",
    "No effect on amounts; it would count as a document without value.",
    "Low", "Section 3 (parents without children)", "Keep as a Document with no lines.",
    affected=[("BKPF", o, BS_KEY)])

### 5.3 Master data

In [31]:
# --- Master data (section 4: SKAT, KNA1, LFA1) ---------------------------------------
skat = tables["SKAT"]
langs = skat.groupby("SAKNR")["SPRAS"].apply(lambda s: ",".join(sorted(s)))
o = skat[skat["SAKNR"].isin(langs[langs != "E,S"].index)]
log_issue("SKAT", "SPRAS", "Completeness",
    "Every account must have its text in both languages, E and S (README: text languages)",
    f"Account {o['SAKNR'].iloc[0]} ({o['TXT50'].iloc[0]}) only has an English text.",
    len(o), skat["SAKNR"].nunique(),
    "The knowledge graph needs labels in both languages; an account without the Spanish text will show no Spanish label.",
    "Cosmetic: affects the label of one account in Spanish, no effect on amounts.",
    "Low", "Section 4.3 (SKAT)", "Add the Spanish text, or fall back to the English label.",
    rule_source="README (labels in English and Spanish)",
    affected=[("SKAT", o, ["KTOPL", "SAKNR", "SPRAS"])])

kna1 = tables["KNA1"]
o = kna1[~kna1["LAND1"].str.fullmatch(r"[A-Z]{2}")]
log_issue("KNA1", "LAND1", "Validity",
    "Country must be a two-letter uppercase code (README: identifiers and codes follow the system format)",
    f"{len(o)} customers have the country in lowercase ('{o['LAND1'].iloc[0]}' instead of '{o['LAND1'].iloc[0].upper()}').",
    len(o), len(kna1),
    "'es' and 'ES' are different strings: grouping or filtering by country treats them as two countries, and in the graph they become two country individuals.",
    "Low: 2 customers; it only breaks country-based grouping.",
    "Low", "Section 4.8 (KNA1)", "Normalize to uppercase.",
    rule_source="README (identifiers and codes follow the system format)",
    affected=[("KNA1", o, ["KUNNR"])])

o = kna1[kna1["KTOKD"] == "CPD"]
l = bseg[bseg["KUNNR"].isin(o["KUNNR"])].merge(bkpf[BS_KEY + ["BLART"]], on=BS_KEY)
op = l[l["AUGBL"] == ""]
kinds = {"DR": ("invoice", "invoices"), "DZ": ("payment", "payments"), "DG": ("credit memo", "credit memos")}
mix = ", ".join(f"{n} {kinds[t][n > 1]}" for t, n in l["BLART"].value_counts().sort_index().items())
open_txt = "; ".join(f"document {r.BELNR} (type {r.BLART}, {r.WRBTR} {'credit' if r.SHKZG == 'H' else 'debit'} on the customer, not cleared)" for r in op.itertuples())
log_issue("KNA1", "KTOKD / STCD1", "Completeness",
    "Receivables are analysed per identifiable customer; the README only defines CPD as a one-time customer",
    f"{len(o)} customers are one-time accounts (CPD, name '{o['NAME1'].iloc[0]}', no Tax ID, city or postal code; countries {', '.join(o['LAND1'])}); they have {len(l)} BSEG lines ({mix}), {len(op)} of them open: {open_txt}.",
    len(o), len(kna1),
    "A one-time account is a generic record shared by many different buyers, so the buyers cannot be told apart, identified by Tax ID or merged. The missing Tax ID is consistent with the account group, but the README does not state it as a rule.",
    "The open item of a generic account would be shown as if it belonged to one customer, although the buyer behind it is unknown.",
    "Low", "Section 4.8 (KNA1)", "Model as OneTimeCustomer and report it apart from identifiable customers.",
    nature="Condition to handle", rule_source="Business rule: one entry per identifiable customer (not stated in the README)",
    affected=[("KNA1", o, ["KUNNR"]), ("BSEG", op, BS_KEY + ["BUZEI"])])

o = kna1[kna1["LOEVM"] != ""]
l = bseg[bseg["KUNNR"].isin(o["KUNNR"])]
log_issue("KNA1", "LOEVM", "Consistency",
    "A record flagged for deletion must be handled explicitly: the flag does not block postings (SAP: LOEVM is not SPERR)",
    f"Customer {o['KUNNR'].iloc[0]} ({o['NAME1'].iloc[0]}) is flagged for deletion but has {len(l)} lines in BSEG ({(l['AUGBL'] != '').sum()} cleared, {(l['AUGBL'] == '').sum()} open).",
    len(o), len(kna1),
    "Dropping flagged customers would silently remove real documents that belong to a settled customer.",
    "One invoice and its payment would disappear from payment-time averages if flagged customers are filtered out; no open item is affected.",
    "Low", "Section 4.8 (KNA1)", "Keep the history and expose the flag as a property (isDeleted).",
    nature="Condition to handle", rule_source="SAP behaviour (LOEVM is a deletion flag, not a block)",
    affected=[("KNA1", o, ["KUNNR"]), ("BSEG", l, BS_KEY + ["BUZEI"])])

lfa1 = tables["LFA1"]
norm = lfa1["NAME1"].str.upper().str.replace(r"[^A-Z0-9]", "", regex=True)
o = lfa1[(lfa1["STCD1"] != "") & lfa1.duplicated("STCD1", keep=False)]
l = bseg[bseg["LIFNR"].isin(o["LIFNR"])]
inv = l[l["SHKZG"] == "H"].assign(v=pd.to_numeric(l["DMBTR"])).groupby("LIFNR")["v"].sum()
log_issue("LFA1", "STCD1 + NAME1", "Uniqueness",
    "A vendor must be a single record: one Tax ID, one master record",
    f"Vendors {' and '.join(o['LIFNR'])} ('{o['NAME1'].iloc[0]}' / '{o['NAME1'].iloc[1]}') share Tax ID {o['STCD1'].iloc[0]}, city and postal code; the names only differ in case and punctuation. "
    f"Credit amounts: {inv.iloc[0]:,.2f} and {inv.iloc[1]:,.2f} (company code 1000, EUR); lines {len(l)}.",
    len(o), len(lfa1),
    "It is the same company registered twice (2019 and 2025), so its spend is split between two entities.",
    "Harbor Systems appears as two smaller vendors instead of one larger one in any ranking or total by vendor.",
    "Medium", "Section 4.9 (LFA1)", "Unify by Tax ID: one entity with two identifiers.",
    rule_source="Business rule: one legal entity, one vendor master record",
    affected=[("LFA1", o, ["LIFNR"]), ("BSEG", l, BS_KEY + ["BUZEI"])])

emp = lfa1[lfa1["KTOKK"] == "EMPL"]
l = bseg[bseg["LIFNR"].isin(emp["LIFNR"])].merge(bkpf[BS_KEY + ["BLART"]], on=BS_KEY)
kr = l[(l["BLART"] == "KR") & (l["SHKZG"] == "H")]
local = tables["T001"].set_index("BUKRS")["WAERS"]
by_company = kr.assign(v=pd.to_numeric(kr["DMBTR"])).groupby("BUKRS")["v"].sum()   # one currency per company code
kr_txt = ", ".join(f"{v:,.2f} {local[c]} in company {c}" for c, v in by_company.items())
log_issue("LFA1", "KTOKK / STCD1", "Consistency",
    "The vendor analysis covers suppliers that invoice the company (README: LIEF vendor, EMPL employee for travel expenses)",
    f"{len(emp)} vendor records are employees (EMPL; {(emp['STCD1'] == '').sum()} without Tax ID, all with city and postal code). They have {len(l)} lines in BSEG; their expense claims (type KR) add up to {kr_txt}.",
    len(emp), len(lfa1),
    "Employees are paid their expense claims through the vendor ledger, with the same document type (KR) as real suppliers, but they are people, not suppliers.",
    "Employee expense claims would be ranked together with supplier invoices in any total by vendor.",
    "Low", "Section 4.9 (LFA1)", "Model as Employee, separate from Vendor, and leave them out of supplier totals.",
    nature="Condition to handle", rule_source="README (account group: LIEF vendor, EMPL employee)",
    affected=[("LFA1", emp, ["LIFNR"])])

### 5.4 Document headers

In [32]:
# --- Document headers (section 4: BKPF) ----------------------------------------------
vi = bseg[bseg["KOART"] == "K"].merge(bkpf[BS_KEY + ["BLART", "XBLNR", "BLDAT", "BUDAT", "STBLG"]], on=BS_KEY)
vi = vi[(vi["BLART"] == "KR") & (vi["XBLNR"] != "")].drop_duplicates(BS_KEY)
dup = vi[vi.duplicated(["LIFNR", "XBLNR"], keep=False)]
for (ven, ref), g in dup.groupby(["LIFNR", "XBLNR"]):
    same = g[["WRBTR", "BLDAT"]].drop_duplicates().shape[0] == 1
    pay = bkpf[(bkpf["BUKRS"] == g["BUKRS"].iloc[0]) & bkpf["BELNR"].isin(g.loc[g["AUGBL"] != "", "AUGBL"])]
    paid_twice = (g["AUGBL"] != "").all() and g["AUGBL"].nunique() == len(g)
    paid_txt = (f" Both were paid, by separate payments ({' and '.join(sorted(pay['BELNR']))}, posted on {ymd(pay['BUDAT'].min())} and {ymd(pay['BUDAT'].max())})."
                if paid_twice else "")
    log_issue("BKPF + BSEG", "XBLNR", "Uniqueness",
        "A vendor invoice number is entered once per vendor (LIFNR + XBLNR is unique)",
        f"Vendor {ven} has documents {' and '.join(g['BELNR'])} (company {g['BUKRS'].iloc[0]}, type KR) with the same invoice number {ref}"
        + (f", the same amount ({g['WRBTR'].iloc[0]}) and the same invoice date ({ymd(g['BLDAT'].iloc[0])}); they were posted on {ymd(g['BUDAT'].min())} and {ymd(g['BUDAT'].max())}"
           if same else "") + f"; reversed: {'neither' if (g['STBLG'] == '').all() else 'at least one'}." + paid_txt,
        len(g), len(vi),
        "Same vendor, same invoice number, amount and date, and no reversal: the invoice was entered twice. This is the check SAP itself uses to detect duplicate invoices.",
        "The invoice is counted twice in the vendor's total and " + ("was paid twice." if paid_twice else "could be paid twice."),
        "Medium", "Section 4.10 (BKPF)", "Mark as a probable duplicate and decide with the business whether to exclude one of them from the vendor total.",
        rule_source="Business rule (SAP duplicate-invoice check: vendor + reference)",
        affected=[("BKPF", g, BS_KEY)])

cr = bseg[bseg["KOART"] == "D"].merge(bkpf[BS_KEY + ["BLART", "XBLNR", "BUDAT"]], on=BS_KEY)
cr = cr[(cr["BLART"] == "DR") & (cr["XBLNR"] != "")].drop_duplicates(BS_KEY)
col = cr[cr.duplicated(["BUKRS", "XBLNR"], keep=False)]
for (co, ref), g in col.groupby(["BUKRS", "XBLNR"]):
    log_issue("BKPF + BSEG", "XBLNR", "Uniqueness",
        "A customer invoice number is issued by the company itself and is unique within the company code (BUKRS + XBLNR)",
        f"Reference {ref} (type DR, company {co}) is used by documents {' and '.join(g['BELNR'])}, posted on {ymd(g['BUDAT'].min())} and {ymd(g['BUDAT'].max())}, "
        f"for customers {' and '.join(g['KUNNR'])} and amounts {' and '.join(g['WRBTR'])}. Customers and amounts differ, so it is not a duplicate, but one number identifies two invoices.",
        len(g), len(cr),
        "The company numbers its own invoices, so a repeated number is a numbering error or a mistake when entering the reference. It cannot identify one invoice.",
        "If XBLNR is used as the invoice identifier in the graph, two different invoices would be merged into one node and their receivables would be mixed.",
        "Low", "Section 4.10 (BKPF)", "Identify invoices by BUKRS + BELNR + GJAHR, never by XBLNR; ask the source why the number repeats.",
        rule_source="Business rule (the company numbers its own invoices; not stated in the README)",
        affected=[("BKPF", g, BS_KEY)])

early = bkpf[bkpf["BLDAT"] < "20250101"]
late = bkpf[bkpf["CPUDT"] > "20251231"]
ll = bseg.merge(late[BS_KEY], on=BS_KEY)
late_open = ll[(ll["KOART"] == "D") & (ll["AUGBL"] == "")]
log_issue("BKPF", "BLDAT / BUDAT / CPUDT", "Consistency",
    "Year and key date are measured on one stated reference date (challenge rule: key date 31 Dec 2025; the date field to use is not specified)",
    f"{len(early)} documents (type {', '.join(early['BLART'].unique())}, vendor invoices) have a document date in December 2024 and a posting date in 2025. "
    f"{len(late)} documents posted in December 2025 were entered on {ymd(late['CPUDT'].min())} or {ymd(late['CPUDT'].max())}, after the key date; they leave {len(late_open)} open customer lines and include {(late['BLART'] == 'KR').sum()} vendor invoice.",
    len(early) + len(late), len(bkpf),
    "Three dates describe the same document and fall in different periods for these documents; the date chosen decides which documents belong to 2025 and to the key date.",
    f"{len(early)} vendor invoices drop out of 2025 if the document date is used, and {len(late_open)} open customer lines are missed at the key date if the entry date is used.",
    "Low", "Section 4.10 (BKPF)", "State the assumption: use the posting date (BUDAT) for the year and the key date, and the baseline date (ZFBDT) for aging.",
    nature="Condition to handle", rule_source="Challenge rules (key date) and README (date fields)",
    affected=[("BKPF", early, BS_KEY), ("BKPF", late, BS_KEY)])

rev = bkpf[bkpf["STBLG"] != ""]
orig = rev[rev["BLART"] != "AB"]
ab = rev[rev["BLART"] == "AB"]
l = bseg.merge(rev[BS_KEY], on=BS_KEY)
pair = orig.merge(bkpf[BS_KEY + ["MONAT"]], left_on=["BUKRS", "STBLG", "STJAH"], right_on=BS_KEY, suffixes=("", "_rev"))
shift = pair[pair["MONAT"] != pair["MONAT_rev"]]
shift_txt = "; ".join(f"company {r.BUKRS}, period {r.MONAT} to {r.MONAT_rev}" for r in shift.itertuples())
log_issue("BKPF + BSEG", "STBLG / STJAH", "Consistency",
    "A reversed document and its reversal must not be counted in balances or profit and loss",
    f"{len(rev)} documents are involved in reversals: {len(ab)} reversal documents (type AB, 'Reversal - entry error') and the {len(orig)} originals that point to them ({len(l)} lines in total). "
    "Each original is cleared by its own reversal document, so on its own it still looks like a normal, settled invoice. "
    f"{len(shift)} of the reversals were posted in a later period than their original ({shift_txt}).",
    len(rev), len(bkpf),
    "Original and reversal cancel each other. Keeping only the original overstates revenue or cost; keeping only the reversal understates it; both must be treated together.",
    "Including only the originals overstates vendor invoicing, revenue and costs; the clearing date of an original is the reversal date, not a payment, which would distort days-to-pay measures. Where original and reversal fall in different periods, leaving out only one of them moves the amount from one month to another.",
    "High", "Section 4.10 (BKPF)", "Exclude original and reversal from metrics and model the relationship reversedBy.",
    nature="Condition to handle", rule_source="SAP behaviour (reversal links STBLG / STJAH)",
    affected=[("BKPF", rev, BS_KEY)])

mig = bkpf[bkpf["USNAM"] == "MIGRATION"]
l = bseg.merge(mig[BS_KEY], on=BS_KEY)
pl = tables["SKA1"].set_index("SAKNR")["GVTYP"]
names = tables["SKAT"].query("SPRAS == 'E'").set_index("SAKNR")["TXT50"]
recon = tables["SKB1"].loc[tables["SKB1"]["MITKZ"] != "", "SAKNR"].unique()      # receivables / payables accounts
log_issue("BKPF", "USNAM / BLART", "Accuracy",
    "Opening balances are not business activity and must be treated apart from the year's postings",
    f"{len(mig)} documents of type SA entered by user MIGRATION on {ymd(mig['BUDAT'].min())} ('{mig['BKTXT'].iloc[0]}', one per company code): {len(l)} lines, "
    f"{(pd.to_numeric(l['WRBTR']) > 200000).sum()} with amounts above 200,000, all on balance-sheet accounts ({(l['HKONT'].map(pl) == 'X').sum()} on profit and loss): {', '.join(names[a] for a in sorted(l['HKONT'].unique()))}. "
    f"{l['HKONT'].isin(recon).sum()} lines are on the receivables or payables accounts.",
    len(mig), len(bkpf),
    "They carry the balance brought forward, not transactions of the year; mixing them with activity inflates balances and posting counts.",
    "Cash, fixed-asset and equity balances need them; profit and loss is not affected because they only touch balance-sheet accounts. No receivable or payable is brought forward, so every open customer or vendor item comes from a 2025 document.",
    "High", "Section 4.10 (BKPF)", "Exclude from profit and loss; include in balances.",
    nature="Condition to handle", rule_source="Business rule: opening balances are not business activity",
    affected=[("BKPF", mig, BS_KEY)])

### 5.5 Line items

In [33]:
# --- Line items (section 4: BSEG) ----------------------------------------------------
text = tables["SKAT"].query("SPRAS == 'E'").set_index("SAKNR")["TXT50"]
o = bseg[pd.to_numeric(bseg["WRBTR"]) == 0]
acc = o.groupby("HKONT").size()
m = bseg.merge(bkpf[BS_KEY + ["KURSF", "WAERS"]], on=BS_KEY)
m = m[pd.to_numeric(m["WRBTR"]) != 0]
expected = pd.to_numeric(m["WRBTR"]) * m["KURSF"].astype(float)
gap = (pd.to_numeric(m["DMBTR"]) - expected).abs()
off = m[gap > 0.02]                                  # anything beyond rounding
over_1pct = (gap[off.index] / expected[off.index] > 0.01).sum()
off_docs = off[BS_KEY].drop_duplicates()
covered = off_docs.merge(o[BS_KEY].drop_duplicates(), on=BS_KEY)
log_issue("BSEG", "WRBTR / DMBTR", "Consistency",
    "The local amount follows the document amount (DMBTR = WRBTR x BKPF.KURSF); the amounts are positive and the sign comes from SHKZG (README)",
    f"{len(o)} lines have a document amount of 0 but a local amount: "
    + ", ".join(f"{a} ({text[a]}): {n}" for a, n in acc.items())
    + f". They are exchange-rate differences. Separately, {len(off)} customer/vendor lines in {len(off_docs)} payment documents ({', '.join(sorted(off['WAERS'].unique()))}) have a DMBTR that is not WRBTR x KURSF "
    f"({over_1pct} of them by more than 1 %); {len(covered)} of those {len(off_docs)} documents contain one of the exchange-rate lines, which carries the difference and balances the document.",
    len(o), len(bseg),
    "The local amount cannot always be derived from WRBTR x rate; the difference sits on separate exchange-rate lines. SAP produces them when an open item is cleared at a different rate, so the data is correct, but the two columns are not interchangeable.",
    "Summing WRBTR leaves these gains and losses out of profit and loss, and recomputing the local amount as WRBTR x KURSF gives a different figure on the payments settled at another rate.",
    "Medium", "Section 4.11 (BSEG)", "Use DMBTR as the local-currency amount and never recompute it from WRBTR x KURSF.",
    nature="Condition to handle", rule_source="SAP behaviour (clearing at a different exchange rate)",
    affected=[("BSEG", o, BS_KEY + ["BUZEI"]), ("BSEG", off, BS_KEY + ["BUZEI"])])

fs = bseg.merge(tables["SKB1"][["BUKRS", "SAKNR", "FSTAG"]], left_on=["BUKRS", "HKONT"], right_on=["BUKRS", "SAKNR"])
g4 = fs[fs["FSTAG"] == "G004"]
o = g4[g4["KOSTL"] == ""]
log_issue("BSEG", "KOSTL", "Completeness",
    "Lines on accounts with field status group G004 require a cost center (README: FSTAG)",
    f"{len(o)} expense lines have no cost center, on accounts {', '.join(sorted(o['HKONT'].unique()))} (vendor invoices, type {', '.join(o['BSCHL'].unique())}).",
    len(o), len(g4),
    "The configuration says a cost center is mandatory, yet these costs were posted without one, so nobody owns them.",
    "These costs cannot be assigned to any cost center, so the totals per cost center are incomplete.",
    "Medium", "Section 4.11 (BSEG)", "Assign them to a 'No cost center' member and report them apart.",
    rule_source="SAP configuration (SKB1.FSTAG = G004) and README",
    affected=[("BSEG", o, BS_KEY + ["BUZEI"])])

csks = tables["CSKS"]
cc = bseg[bseg["KOSTL"] != ""].merge(bkpf[BS_KEY + ["BUDAT"]], on=BS_KEY).merge(csks[["KOSTL", "DATBI", "LTEXT"]], on="KOSTL")
o = cc[cc["BUDAT"] > cc["DATBI"]]
nxt = csks[(csks["BUKRS"] == o["BUKRS"].iloc[0]) & (csks["DATAB"] > o["DATBI"].iloc[0])]   # cost centers opened after the expiry
nxt_txt = "; ".join(f"{r.KOSTL} ({r.LTEXT}) starts on {ymd(r.DATAB)}" for r in nxt.itertuples())
log_issue("BSEG + CSKS", "KOSTL", "Validity",
    "A cost center must be valid on the posting date (CSKS.DATAB <= BUDAT <= CSKS.DATBI)",
    f"{len(o)} lines were posted to cost center {o['KOSTL'].iloc[0]} ({o['LTEXT'].iloc[0]}) between {ymd(o['BUDAT'].min())} and {ymd(o['BUDAT'].max())}, after it expired on {ymd(o['DATBI'].iloc[0])} "
    f"({pd.to_numeric(o['DMBTR']).sum():,.2f} EUR). In the same company code, cost center {nxt_txt}.",
    len(o), len(cc),
    "The cost center was closed, but postings kept using it. Another cost center with a different name starts the next day; that it replaces the old one is an assumption, the data does not say so.",
    "The old cost center has costs after its end date; if the new one is its replacement, the cost of the store is split between two cost centers.",
    "Low", "Section 4.5 (CSKS) and 4.11 (BSEG)", "Keep the postings and expose the validity dates; ask whether they belong to the new cost center.",
    rule_source="Master data validity (CSKS.DATAB / DATBI)",
    affected=[("BSEG", o, BS_KEY + ["BUZEI"])])

cl = (bseg[(bseg["KUNNR"] != "") & (bseg["ZTERM"] != "")].merge(bkpf[BS_KEY + ["BLART"]], on=BS_KEY)
      .merge(tables["KNA1"][["KUNNR", "ZTERM"]], on="KUNNR", suffixes=("_line", "_master")))
o = cl[cl["ZTERM_line"] != cl["ZTERM_master"]]
pairs = o.groupby(["ZTERM_line", "ZTERM_master"]).size()
memos = cl[cl["BLART"] == "DG"]
invoices = cl[cl["BLART"] == "DR"]
log_issue("BSEG + KNA1", "ZTERM", "Consistency",
    "The payment terms on a customer line normally follow the customer master (BSEG.ZTERM = KNA1.ZTERM)",
    f"{len(o)} customer lines have different payment terms from the master, and all of them are credit memos (type {', '.join(o['BLART'].unique())}): "
    f"the {len(memos)} credit memos always carry {', '.join(memos['ZTERM_line'].unique())} on the line, whatever the master says ("
    + ", ".join(f"line {a} vs master {b}: {n}" for (a, b), n in pairs.items()) + f"; customers affected: {o['KUNNR'].nunique()}). "
    f"{(invoices['ZTERM_line'] != invoices['ZTERM_master']).sum()} of the {len(invoices)} invoice lines (type DR) differ from their master.",
    len(o), len(cl),
    "A credit memo is posted as due immediately; its terms describe the credit, not a change in what was agreed with the customer. Reading a customer's terms from all of its lines would show two different terms for the same customer.",
    f"The terms agreed with these customers look ambiguous if credit memos are included, and the due date of a credit memo is simply its baseline date; {(o['AUGBL'] == '').sum()} of the {len(o)} lines are open items at the key date.",
    "Low", "Section 4.11 (BSEG)", "Use the terms on the line to date each item, and take the terms agreed with the customer from the master, which always matches the invoice lines.",
    nature="Condition to handle", rule_source="Business rule (not stated in the README)",
    affected=[("BSEG", o, BS_KEY + ["BUZEI"])])

# Open credit memos
cu = bseg[bseg["KOART"] == "D"].merge(bkpf[BS_KEY + ["BLART", "XBLNR", "WAERS"]], on=BS_KEY)
open_cust = cu[cu["AUGBL"] == ""]
memos = cu[cu["BLART"] == "DG"]
o = memos[memos["AUGBL"] == ""]
tot = o.assign(v=pd.to_numeric(o["WRBTR"])).groupby(["BUKRS", "WAERS"])["v"].sum()
inv = cu.loc[cu["BLART"] == "DR", ["BUKRS", "BELNR", "KUNNR", "AUGBL"]].rename(
    columns={"BELNR": "XBLNR", "KUNNR": "KUNNR_inv", "AUGBL": "AUGBL_inv"})
ref = o.merge(inv, on=["BUKRS", "XBLNR"], how="left")           # XBLNR of a credit memo holds the invoice number
same = ref[ref["KUNNR"] == ref["KUNNR_inv"]]
log_issue("BKPF + BSEG", "BLART / AUGBL / XBLNR", "Consistency",
    "A credit memo reduces what the customer owes and is expected to be applied to an invoice or refunded, which clears it",
    f"{len(o)} of the {len(memos)} customer credit memos (type DG) are open at the key date: "
    + ", ".join(f"{v:,.2f} {w} in company {c}" for (c, w), v in tot.items())
    + f". Each one gives in XBLNR the number of the invoice it relates to ({len(same)} of {len(ref)} match an invoice of the same customer); "
    f"{(same['AUGBL_inv'] != '').sum()} of those invoices were already cleared by a payment and {(same['AUGBL_inv'] == '').sum()} are still open.",
    len(o), len(open_cust),
    "An open credit memo is a receivable with the opposite sign. Where its invoice was already paid there is nothing left to offset it against, so it is money owed to the customer.",
    "Leaving the credit memos out overstates what customers owe; netting them can leave a customer with a negative balance; and a credit has no meaningful debt age, because it is due on its baseline date.",
    "Medium", "Section 4.11 (BSEG)", "Keep them as open items with the opposite sign, net them in the customer's balance, show them apart from the age buckets and link each one to its invoice through XBLNR.",
    nature="Condition to handle", rule_source="Business rule (credit memos reduce receivables; not stated in the README)",
    affected=[("BSEG", o, BS_KEY + ["BUZEI"])])

# No exchange-rate table: rates to euros can only be rebuilt from KURSF
t001 = tables["T001"]
h = bkpf.merge(t001[["BUKRS", "WAERS"]].rename(columns={"WAERS": "LOCAL"}), on="BUKRS")
fx = h[h["WAERS"] != h["LOCAL"]].assign(month=lambda d: d["BUDAT"].str[4:6], rate=lambda d: d["KURSF"].astype(float))
months = sorted(bkpf["BUDAT"].str[4:6].unique())

def quoted(a, b):
    """Months with at least one document giving a rate between currencies a and b, in either direction."""
    q = fx[((fx["WAERS"] == a) & (fx["LOCAL"] == b)) | ((fx["WAERS"] == b) & (fx["LOCAL"] == a))]
    return set(q["month"])

non_eur = t001[t001["WAERS"] != "EUR"]
parts = []
for r in non_eur.itertuples():
    others = [c for c in non_eur["WAERS"] if c != r.WAERS]
    direct = quoted(r.WAERS, "EUR")
    missing = [mth for mth in months if mth not in direct]
    cross = [mth for mth in missing if any(mth in quoted(r.WAERS, c) and mth in quoted(c, "EUR") for c in others)]
    parts.append(f"{r.WAERS}/EUR (company {r.BUKRS}) is quoted in {len(direct)} of {len(months)} months"
                 + (f" (missing: {', '.join(missing)}, " + ("which can be obtained" if len(cross) == len(missing) else f"{len(cross)} of which can be obtained")
                    + f" by crossing through {' or '.join(others)})" if missing else ""))
spread = fx.groupby(["WAERS", "LOCAL", "month"])["rate"].agg(lambda s: (s.max() - s.min()) / s.mean() * 100).max()
log_issue("T001 + BKPF", "WAERS / KURSF", "Completeness",
    "Figures for the whole group must be in euros, and the challenge provides no euro exchange-rate table: the rates have to be derived from the data and the method explained",
    f"The extract has no exchange-rate table. Company codes {' and '.join(non_eur['BUKRS'])} keep their books in {' and '.join(non_eur['WAERS'])}: "
    f"{bseg['BUKRS'].isin(non_eur['BUKRS']).sum()} of {len(bseg)} lines have a local amount that is not in euros. The only rates available are the KURSF of {len(fx)} foreign-currency documents: "
    + "; ".join(parts) + f". Within one month the rate of a currency pair varies by up to {spread:.2f} %.",
    len(non_eur), len(t001),
    "DMBTR is in the currency of each company code, so it is in euros only for company 1000. The rates have to be rebuilt from documents that happened to be posted in a foreign currency, which leaves gaps and several rates for the same month.",
    "Every euro figure for these two company codes depends on a derived rate: it changes with the method chosen (monthly average, rate of the day, cross rate) and is an approximation in the months with no direct quote.",
    "High", "Section 4.10 (BKPF)", "Build a monthly rate table from KURSF (average per month, inverse and cross rates where there is no direct quote), keep the method used for each month, and convert DMBTR with it.",
    nature="Condition to handle", rule_source="Challenge rules (no euro exchange-rate table is provided; derive the rates from the data and explain the method)",
    affected=[("T001", non_eur, ["BUKRS"])])


### 5.6 The log

In [34]:
log

,id,table,column,dimension,rule_broken,rule_source,nature,finding,affected_rows,total_rows,pct_affected,why_it_matters,impact,impact_level,evidence,proposed_action
0,DQ-01,BSEG,KUNNR,Referential integrity,Every customer used on a line must exist in th...,Referential integrity,Defect,Customer 0000100999 is used on 3 lines but doe...,3,1299,0.23,Without the master record the customer has no ...,These 3 lines cannot be assigned to a known cu...,High,Section 3 and 3.1,Keep the lines and flag them as orphan custome...
1,DQ-02,BSEG,BUKRS + HKONT,Referential integrity,Every account posted in a company code must be...,Referential integrity,Defect,Account 0000800100 (Revenue - services) is pos...,2,6062,0.03,SKB1 holds the company-level definition of the...,Revenue of company 3000 sits on an account wit...,Medium,Section 3.1 and 4.4 (SKB1),Keep the lines; confirm with the business whet...
2,DQ-03,SKAT,KTOPL + SAKNR,Referential integrity,Every account text must belong to an account o...,Referential integrity,Defect,Account 0000455000 has texts in English and Sp...,2,55,3.64,A text with no account is a label for somethin...,"No effect on amounts, because the account is n...",Low,Section 3.1,"Do not load as GLAccount, or load it flagged a..."
3,DQ-04,BKPF,BELNR,Referential integrity,Every document header must have at least one l...,Referential integrity,Defect,Document 0100000050 ('Accrual Q4 - pending') h...,1,2353,0.04,A posted document without lines carries no amo...,No effect on amounts; it would count as a docu...,Low,Section 3 (parents without children),Keep as a Document with no lines.
4,DQ-05,SKAT,SPRAS,Completeness,Every account must have its text in both langu...,README (labels in English and Spanish),Defect,Account 0000499000 (Miscellaneous expense) onl...,1,28,3.57,The knowledge graph needs labels in both langu...,Cosmetic: affects the label of one account in ...,Low,Section 4.3 (SKAT),"Add the Spanish text, or fall back to the Engl..."
5,DQ-06,KNA1,LAND1,Validity,Country must be a two-letter uppercase code (R...,README (identifiers and codes follow the syste...,Defect,2 customers have the country in lowercase ('es...,2,156,1.28,'es' and 'ES' are different strings: grouping ...,Low: 2 customers; it only breaks country-based...,Low,Section 4.8 (KNA1),Normalize to uppercase.
6,DQ-07,KNA1,KTOKD / STCD1,Completeness,Receivables are analysed per identifiable cust...,Business rule: one entry per identifiable cust...,Condition to handle,"3 customers are one-time accounts (CPD, name '...",3,156,1.92,A one-time account is a generic record shared ...,The open item of a generic account would be sh...,Low,Section 4.8 (KNA1),Model as OneTimeCustomer and report it apart f...
7,DQ-08,KNA1,LOEVM,Consistency,A record flagged for deletion must be handled ...,"SAP behaviour (LOEVM is a deletion flag, not a...",Condition to handle,Customer 0000100023 (Harbor Comercial S.A.) is...,1,156,0.64,Dropping flagged customers would silently remo...,One invoice and its payment would disappear fr...,Low,Section 4.8 (KNA1),Keep the history and expose the flag as a prop...
8,DQ-09,LFA1,STCD1 + NAME1,Uniqueness,"A vendor must be a single record: one Tax ID, ...","Business rule: one legal entity, one vendor ma...",Defect,Vendors 0000200004 and 0000200072 ('Harbor Sys...,2,72,2.78,It is the same company registered twice (2019 ...,Harbor Systems appears as two smaller vendors ...,Medium,Section 4.9 (LFA1),Unify by Tax ID: one entity with two identifiers.
9,DQ-10,LFA1,KTOKK / STCD1,Consistency,The vendor analysis covers suppliers that invo...,"README (account group: LIEF vendor, EMPL emplo...",Condition to handle,6 vendor records are employees (EMPL; 6 withou...,6,72,8.33,Employees are paid their expense claims throug...,Employee expense claims would be ranked togeth...,Low,Section 4.9 (LFA1),"Model as Employee, separate from Vendor, and l..."


### 5.7 Reviewed and found normal
These observations were checked against a rule and **do not break it**, so they are not in the log. They are kept here so a reviewer can see they were examined and why they were accepted.

In [35]:
# --- Reviewed and found normal (not logged as problems) --------------------------------
kna1, lfa1, t001 = tables["KNA1"], tables["LFA1"], tables["T001"]

l = bseg[bseg["KUNNR"] != ""].merge(kna1[["KUNNR", "KTOKD", "LAND1"]], on="KUNNR").merge(
    t001[["BUKRS", "LAND1"]].rename(columns={"LAND1": "COMPANY_LAND"}), on="BUKRS")
domestic = l["LAND1"].str.upper() == l["COMPANY_LAND"]
bad = ((l["KTOKD"] == "KUNA") & ~domestic) | ((l["KTOKD"] == "KUNX") & domestic)
accept("KNA1 + BSEG + T001", "KTOKD / LAND1",
    f"Domestic (KUNA) and foreign (KUNX) customers are defined relative to the selling company's country: {bad.sum()} of {l['KTOKD'].isin(['KUNA', 'KUNX']).sum()} KUNA/KUNX lines contradict it.",
    bad.sum(), "KNA1 has no company code, so a customer in GB or US is domestic for company 3000 or 2000. Compared against T001.LAND1 the account groups are consistent.")

o = bseg[bseg["ZFBDT"] != ""].merge(bkpf[BS_KEY + ["BLART", "BLDAT"]], on=BS_KEY)
o = o[o["ZFBDT"] < o["BLDAT"]]
accept("BSEG", "ZFBDT",
    f"{len(o)} lines have a baseline date before the document date; all are in type {', '.join(o['BLART'].unique())} (reversals).",
    len(o), "A reversal inherits the baseline date of the document it cancels. Both are excluded from the metrics (see the reversal entry).")

abbr = ["cogs", "mkt", "it", "rent", "cons", "trav", "off", "util"]
o = bseg[bseg["SGTXT"].isin(abbr)]
accept("BSEG", "SGTXT",
    f"{len(o)} lines have an abbreviated code as text ({', '.join(abbr)}); {(bseg['SGTXT'] == '').sum()} lines have no text.",
    len(o), "SGTXT is free text with no defined format in the README, and the type of expense comes from the G/L account (HKONT), so nothing depends on it.")

b = bseg.assign(s=pd.to_numeric(bseg["DMBTR"]).where(bseg["SHKZG"] == "S", -pd.to_numeric(bseg["DMBTR"])))
unbalanced = (b.groupby(BS_KEY)["s"].sum().round(2).abs() > 0.01).sum()
accept("BSEG", "DMBTR / SHKZG",
    f"Every document balances: debits equal credits in local currency; {unbalanced} documents do not.",
    unbalanced, "This confirms that SHKZG and DMBTR are consistent with each other.")

nm = bseg.merge(bkpf[BS_KEY + ["USNAM"]], on=BS_KEY)
nm = nm[nm["USNAM"] != "MIGRATION"]
accept("BSEG", "WRBTR",
    f"The largest document amount outside the opening balances is {pd.to_numeric(nm['WRBTR']).max():,.2f}; the amounts above 200,000 are all in the opening-balance documents.",
    (pd.to_numeric(nm["WRBTR"]) > 200000).sum(), "No amount outliers in the year's activity.")

o = bseg[bseg["KOART"].isin(["D", "K"]) & (bseg["AUGBL"] == "")].merge(bkpf[BS_KEY + ["BLART"]], on=BS_KEY)
cust = o[o["KOART"] == "D"]
latest = bseg.loc[bseg["AUGDT"] != "", "AUGDT"].max()
accept("BSEG", "AUGBL",
    f"{len(o)} customer/vendor lines have no clearing document at the end of the year ({len(cust)} customer: {(cust['BLART'] == 'DR').sum()} invoices and {(cust['BLART'] == 'DG').sum()} credit memos; {(o['KOART'] == 'K').sum()} vendor); the latest clearing date in the data is {ymd(latest)}, so none was cleared after the key date.",
    len(o), "An empty clearing document (AUGBL) is how SAP marks an open item; it is not an error. It is the definition of the open items (receivables and payables) at the key date, so it is stated here as an assumption. The open credit memos have their own entry in the log.")

reviewed

,table,column,observation,rows,why_it_is_normal
0,KNA1 + BSEG + T001,KTOKD / LAND1,Domestic (KUNA) and foreign (KUNX) customers a...,0,"KNA1 has no company code, so a customer in GB ..."
1,BSEG,ZFBDT,7 lines have a baseline date before the docume...,7,A reversal inherits the baseline date of the d...
2,BSEG,SGTXT,484 lines have an abbreviated code as text (co...,484,SGTXT is free text with no defined format in t...
3,BSEG,DMBTR / SHKZG,Every document balances: debits equal credits ...,0,This confirms that SHKZG and DMBTR are consist...
4,BSEG,WRBTR,The largest document amount outside the openin...,0,No amount outliers in the year's activity.
5,BSEG,AUGBL,167 customer/vendor lines have no clearing doc...,167,An empty clearing document (AUGBL) is how SAP ...


### 5.8 Save

In [36]:
os.makedirs("data_quality", exist_ok=True)
log.to_csv("data_quality/data_quality_log.csv", index=False, encoding="utf-8-sig")
# One line per affected row, each with the full description of its finding (rule, why, impact, action)
summary_only = ["table", "affected_rows", "total_rows", "pct_affected"]
detail_full = detail.merge(log.drop(columns=summary_only), on="id")
detail_full.to_csv("data_quality/data_quality_detail.csv", index=False, encoding="utf-8-sig")
reviewed.to_csv("data_quality/reviewed_and_accepted.csv", index=False, encoding="utf-8-sig")
print(len(log), "entries saved to data_quality/data_quality_log.csv;", len(detail_full), "affected rows saved to data_quality/data_quality_detail.csv;", len(reviewed), "reviewed items saved to data_quality/reviewed_and_accepted.csv")

21 entries saved to data_quality/data_quality_log.csv; 200 affected rows saved to data_quality/data_quality_detail.csv; 6 reviewed items saved to data_quality/reviewed_and_accepted.csv
